https://huggingface.co/meta-llama/Llama-3.2-3B

In [ ]:
!git clone https://github.com/ryy1210/RMT_utils

import sys
sys.path.append("/content/RMT_utils")

!pip install weightwatcher
import funcs1
import evaluater

from evaluater import ppl_eval


from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

# import wandb
# wandb.login()



In [ ]:
%cd /content/RMT_utils
!git pull
%cd /content

# モデルの読み込み

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM
from google.colab import userdata

# 1. シークレットからトークンを読み込んで環境変数にセット
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

# 本家 Llama-3.2-3B のモデルIDを指定
model_id = "meta-llama/Llama-3.2-3B"

print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu" # 念のため一度CPUに展開
)

# 2. ターゲット層の抽出
# 例として、中間層（Layer 14）の Self-Attention における q_proj を抽出します
layer_idx = 14
target_layer = model.model.layers[layer_idx].self_attn.q_proj

# 重みテンソルを取得し、NumPy配列に変換
W = target_layer.weight.detach().cpu().numpy().astype(np.float32)
print(f"\n[成功] 重み行列を抽出しました (Shape: {W.shape})")
print(f"  - 出力次元 (out_features): {W.shape[0]}")
print(f"  - 入力次元 (in_features): {W.shape[1]}")

In [ ]:
import pandas as pd
from google.colab import drive
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM
from google.colab import userdata

# 1. Google Drive のマウント
drive.mount('/content/drive')

# 保存先のディレクトリを作成（ご自身の環境に合わせて変更してください）
save_dir = '/content/drive/MyDrive/TUS/hashiguchi/data'
os.makedirs(save_dir, exist_ok=True)

filename = "llama-3.2-3B_esd_metrics.pkl"
filepath = os.path.join(save_dir, filename)

if os.path.exists(filepath):
    results = pd.read_pickle(filepath)
    print(f"✅ データの読み込みが完了しました: {filepath}")
    print(f"行数（層の数）: {len(results)}")
else:
    print(f"❌ ファイルが見つかりません: {filepath}")

# ESD

In [ ]:
import pandas as pd
# 1. 対象の層の重みを抽出
layer_idx = 1
target_layer = model.model.layers[layer_idx].self_attn.q_proj
W = target_layer.weight.detach().cpu().numpy().astype(np.float32)


# 3. CSVとして保存 (ヘッダーやインデックスは不要なので False)
# Colab環境なら '/content/llama_weight.csv' 等に保存します
save_path = "llama_weight.csv"
print(f"CSVへの書き出しを開始します... (約150MBになります)")
pd.DataFrame(W).to_csv(save_path, index=False, header=False)
print(f"保存完了: {save_path}")

In [ ]:
# 3. 特異値分解 (SVD) とランダム行列理論(RMT)に基づく固有値計算
print("\n特異値分解（SVD）を計算しています...")
_, S, _ = np.linalg.svd(W, full_matrices=False)

# 相関行列 X = (1/N) * W^T * W の固有値に変換するため、特異値を2乗してサンプリング次元（行数）で割る
# これにより西川氏の論文およびMarchenko-Pastur（MP）分布の理論曲線と完全に一致します
eigenvalues = (S ** 2) / W.shape[0]

print(f"計算された固有値の総数: {len(eigenvalues)}")
print(f"最大固有値 ($\lambda_{{max}}$): {np.max(eigenvalues):.6f}")
print(f"最小固有値 ($\lambda_{{min}}$): {np.min(eigenvalues):.6f}")

# 4. ESD (経験的スペクトル密度) のプロット
plt.figure(figsize=(10, 6))

# 確率密度関数として正規化（density=True）
# plt.hist(eigenvalues, bins=100, density=True, alpha=0.6, color='royalblue', edgecolor='black', label="Llama-3.2 ESD")
plt.hist(eigenvalues, bins=100, density=False, alpha=0.6, color='royalblue', edgecolor='black', label="Llama-3.2 ESD")

plt.title(f"Empirical Spectral Density (ESD) - Llama-3.2-3B Layer {layer_idx} q_proj", fontsize=13)
plt.xlabel("Eigenvalue $\lambda$", fontsize=11)
plt.ylabel("Probability Density $P(\lambda)$", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

plt.show()

In [ ]:
# 4. ESD (経験的スペクトル密度) のプロット
plt.figure(figsize=(10, 6))

# 確率密度関数として正規化（density=True）
# plt.hist(eigenvalues, bins=100, density=True, alpha=0.6, color='royalblue', edgecolor='black', label="Llama-3.2 ESD")
plt.hist(S, bins=100, density=False, alpha=0.6, color='royalblue', edgecolor='black', label="Llama-3.2 ESD")

plt.title(f"Empirical Spectral Density (ESD) - Llama-3.2-3B Layer {layer_idx} q_proj", fontsize=13)
plt.xlabel("Singular $\lambda$", fontsize=11)
plt.ylabel("Probability Density $P(\lambda)$", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

plt.show()

In [ ]:
results['mp_soft_rank_preDE'] = results['threshold_preDE'] / results['eigs'].apply(np.max)

In [ ]:
results[results['alpha'] == np.min(results['alpha'])]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def plot_esd_full_analysis(layer_name, evals, alpha):
    """
    3種類のプロットを並べて表示:

    (1) 通常スケールのESD（linear）
    (2) eigs全体のlog-logヒストグラム
    (3) tail部分 + power-law fit
    """
    evals = np.sort(evals)
    xmax = np.max(evals)

    print('固有値の総数:', len(evals))
    print('最大固有値:', xmax)
    print('最小固有値:', np.min(evals))

    # ===============================
    # 0. データ整形
    # ===============================
    if isinstance(evals, torch.Tensor):
        evals_np = evals.detach().cpu().numpy()
    else:
        evals_np = np.asarray(evals)

    evals_np = evals_np.astype(float)
    evals_np = evals_np[np.isfinite(evals_np)]
    evals_np = evals_np[evals_np > 0]

    if len(evals_np) < 2:
        print("固有値が少なすぎます")
        return

    # ===============================
    # 1. xmin決定 alpha計算のfix-finger法と整合するように
    # ===============================
    nz_eigs = evals_np[evals_np > 1e-8] # EVALS_THRESH の代用
    N = len(nz_eigs)
    if N == 0:
        nz_eigs = evals_np
        N = len(nz_eigs)

    hist, bin_edges = np.histogram(nz_eigs, bins=100)
    peak_bin_idx = np.argmax(hist)

    # ピークとなるビンの左端を閾値 xmin とみなす
    xmin_val = bin_edges[peak_bin_idx]

    # nz_eigsは昇順なので、xmin_val以上の最初のインデックス i を取得
    i = np.searchsorted(nz_eigs, xmin_val).item()

    # 全てがノイズとして切り捨てられないよう、最低限の要素数を確保する安全弁
    if i >= N - 2:
        i = N - 3

    xmin = nz_eigs[i]

    print('xmin：', xmin)

    tail_evals = evals_np[evals_np >= xmin]

    print('tailに含まれる固有値数：', len(tail_evals))

    # ===============================
    # 2. 全体log-log用
    # ===============================
    bins_all = np.logspace(
        np.log10(evals_np.min()),
        np.log10(evals_np.max()),
        80
    )

    counts_all, bin_edges_all = np.histogram(
        evals_np, bins=bins_all, density=True
    )

    centers_all = np.sqrt(bin_edges_all[:-1] * bin_edges_all[1:])
    valid_all = counts_all > 0

    # ===============================
    # 3. tail用
    # ===============================
    bins_tail = np.logspace(
        np.log10(tail_evals.min()),
        np.log10(tail_evals.max()),
        50
    )

    counts_tail, bin_edges_tail = np.histogram(
        tail_evals, bins=bins_tail, density=True
    )

    centers_tail = np.sqrt(bin_edges_tail[:-1] * bin_edges_tail[1:])
    valid_tail = counts_tail > 0

    # ===============================
    # 4. Power-law fit
    # ===============================
    # PLの正規化定数
    if alpha > 1:
        C = (alpha - 1) * (xmin ** (alpha - 1)) #xmin \to \inftyでの正規化定数
        # C = (alpha - 1) / (xmin**(1 - alpha) - xmax**(1 - alpha)) # xmin \to xmaxでの正規化定数
    else:
        C = counts_tail[valid_tail][0] * (centers_tail[valid_tail][0] ** alpha)

    x_fit = np.logspace(
        np.log10(tail_evals.min()),
        np.log10(tail_evals.max()),
        100
    )
    y_fit = C * (x_fit ** (-alpha))

    # ===============================
    # 5. プロット
    # ===============================
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ---------------------------
    # (1) 通常ESD（linear）
    # ---------------------------
    axes[0].hist(
        evals_np,
        bins=50,
        density=True,
        color='gray',
        alpha=0.7
    )

    axes[0].axvline(
        xmin,
        color='red',
        linestyle='--',
        label=f"xmin={xmin:.2e}"
    )

    axes[0].set_title(f"Linear-scale ESD '{layer_name}'")
    axes[0].set_xlabel("Eigenvalue λ")
    axes[0].set_ylabel("Density")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # ---------------------------
    # (2) 全体 log-log
    # ---------------------------
    axes[1].loglog(
        centers_all[valid_all],
        counts_all[valid_all],
        'o',
        color='black',
        alpha=0.7
    )

    axes[1].axvline(xmin, color='red', linestyle='--')

    axes[1].set_title("Full ESD (log-log)")
    axes[1].set_xlabel("λ")
    axes[1].set_ylabel("P(λ)")
    axes[1].grid(True, which="both", ls="--", alpha=0.3)

    # ---------------------------
    # (3) tail + fit
    # ---------------------------
    axes[2].loglog(
        centers_tail[valid_tail],
        counts_tail[valid_tail],
        'o',
        label="Empirical tail"
    )

    axes[2].loglog(
        x_fit,
        y_fit,
        'r--',
        linewidth=2,
        label=f"slope = - alpha = {-alpha:.2f}"
    )

    axes[2].axvline(xmin, color='gray', linestyle=':')

    axes[2].set_title("Tail + Power-law fit")
    axes[2].set_xlabel("λ")
    axes[2].set_ylabel("P(λ)")
    axes[2].legend()
    axes[2].grid(True, which="both", ls="--", alpha=0.3)

    plt.tight_layout()
    plt.show()

layer_name = "model.layers.0.self_attn.q_proj"
# layer_name = "model.layers.0.mlp.down_proj"

# layer_name = 'model.layers.3.mlp.gate_proj' # alpha = 2.071442 でmax
# layer_name = 'model.layers.3.self_attn.q_proj' #alpha = 1.057561 でmin

evals = results[results['name'] == layer_name]['eigs'].iloc[0]
alpha = results[results['name'] == layer_name]['alpha'].iloc[0]

plot_esd_full_analysis(layer_name, evals, alpha)

In [ ]:
results[results['name'] == layer_name]['mp_soft_rank_preDE']

# Dyson Equalizerの適用

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def dyson_equalizer_algorithm1(Y):
    """
    Landa & Kluger (2024) - Algorithm 1: The Dyson Equalizer
    論文の数式と記法に完全に対応させた実装。

    Input:
        Y: Data matrix (m x n), m <= n
    Returns:
        Y_hat: Normalized data matrix
        x_hat: Row scaling vector
        y_hat: Column scaling vector
    """
    m, n = Y.shape
    if m > n:
        raise ValueError("Input matrix Y must have m <= n. Transpose Y if necessary.")

    # 1: Compute the SVD of Y
    # U: m x m, sigma: m, V_h: n x n
    U, sigma, V_h = np.linalg.svd(Y, full_matrices=True)
    V = V_h.T  # V \in R^{n x n} (右特異ベクトルを列に持つ行列)

    # 2: Set eta as the median singular value of Y
    eta = np.median(sigma)

    # 3: Compute the vectors g_hat^(1) and g_hat^(2)
    # 論文 (3) 式の計算（行列演算で高速化）
    term1 = eta / (sigma**2 + eta**2)
    term2 = term1 - (1 / eta)

    # U は m x m, sigma は要素数 m
    g1_hat = (U**2) @ term1

    # V は n x n. sum は k=1 から m までなので V の最初の m 列を使用
    g2_hat = (1 / eta) + (V[:, :m]**2) @ term2

    # 4: Compute the vectors x_hat and y_hat
    # L1ノルム ||g_hat^(1)||_1 と ||g_hat^(2)||_1 の計算
    g1_norm1 = np.sum(np.abs(g1_hat))
    g2_norm1 = np.sum(np.abs(g2_hat))

    # 論文 (4) 式の計算
    x_hat = (1 / np.sqrt(m - eta * g1_norm1)) * ((1 / g1_hat) - eta)
    y_hat = (1 / np.sqrt(n - eta * g2_norm1)) * ((1 / g2_hat) - eta)

    # 数値的安定性のための安全策（負値の平方根エラー回避）
    x_hat = np.maximum(1e-12, x_hat)
    y_hat = np.maximum(1e-12, y_hat)

    # 5: Form the normalized data matrix Y_hat
    # Y_hat = (D_{x_hat})^{-1/2} Y (D_{y_hat})^{-1/2}
    Y_hat = Y / (np.sqrt(x_hat[:, None]) * np.sqrt(y_hat[None, :]))

    return Y_hat, x_hat, y_hat

# ==========================================
# 動作検証用シミュレーション
# ==========================================
def run_simulation():
    print("もとの重みのSVDを計算")
    m, n = W.shape[0], W.shape[1]
    gamma = m / n
    _, S, _ = np.linalg.svd(W, full_matrices=False)

    # ==========================================
    # Algorithm 1 の適用
    # ==========================================
    print("Algorithm 1 (Dyson Equalizer) を適用中...")
    W_hat, x_hat, y_hat = dyson_equalizer_algorithm1(W)
    print("DE完了")

    # 補正後のESD計算 (特異値 S_hat)
    W_hat_scaled = W_hat / np.sqrt(n) # 実装上の修正　スケールを揃える
    _, S_hat, _ = np.linalg.svd(W_hat_scaled, full_matrices=False)

    # ---------------------------------------------------------
    # 【修正】特異値空間における理論曲線の計算 (MP分布の変数変換)
    # ---------------------------------------------------------
    lambda_plus = (1 + np.sqrt(gamma))**2
    lambda_minus = (1 - np.sqrt(gamma))**2

    # 横軸は特異値の軸 (sigma) として定義する
    sigma_max = np.sqrt(lambda_plus)
    sigma_min = np.sqrt(lambda_minus)
    x_ax = np.linspace(max(0, sigma_min - 0.5), sigma_max + 0.5, 1000)

    # 特異値の2乗 (lambda) がMPのサポート内にあるか判定
    valid_mask = (x_ax >= sigma_min) & (x_ax <= sigma_max)
    rho_singular = np.zeros_like(x_ax)

    # ヤコビアン 2*sigma を掛け合わせた特異値空間の理論式
    sigma_val = x_ax[valid_mask]
    lam_val = sigma_val ** 2
    rho_mp_converted = np.sqrt((lambda_plus - lam_val) * (lam_val - lambda_minus)) / (2 * np.pi * gamma * lam_val)
    rho_singular[valid_mask] = 2 * sigma_val * rho_mp_converted

    # ---------------------------------------------------------
    # プロット
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bins = 100

    # 元の特異値分布
    axes[0].hist(S, bins=bins, density=True, color='salmon', alpha=0.8, label='Empirical $S$')
    axes[0].plot(x_ax, rho_singular, 'k--', lw=2, label='Theoretical Singular Law')
    axes[0].set_title('Original Singular Value Density', fontsize=12)
    axes[0].set_xlabel('Singular Value $\sigma$')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # DE適用後の特異値分布
    axes[1].hist(S_hat, bins=bins, density=True, color='dodgerblue', alpha=0.8, label='Empirical $\hat{S}$')
    axes[1].plot(x_ax, rho_singular, 'k--', lw=2, label='Theoretical Singular Law')
    axes[1].set_title('Dyson Equalizer Corrected Density', fontsize=12)
    axes[1].set_xlabel('Singular Value $\sigma$')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    run_simulation()

# BEMA

In [ ]:
!git clone https://github.com/ryy1210/RMT_utils
import sys
sys.path.append("/content/RMT_utils")

import funcs1

In [ ]:
import numpy as np

# BEMAテスト

m, n = 2000, 2000
gamma = m / n
Z = np.random.randn(m, n)
result_orig = funcs1.bema_algorithm1_from_data(Z, alpha=0.2, beta=0.1)

print("BEMA Algorithm 1")
print(f"sigma^2_hat = {result_orig['sigma2_hat']:.4f}")
print(f"s_hat        = {result_orig['s_hat']}")
print(f"threshold    = {result_orig['threshold']:.4f}")

In [ ]:
# 4分くらい
result = funcs1.get_esd_metrics(model)

In [ ]:
import pandas as pd
import wandb
import pickle
import os
from google.colab import drive

# 1. Google Drive のマウント
drive.mount('/content/drive')

# 保存先のディレクトリを作成（ご自身の環境に合わせて変更してください）
save_dir = '/content/drive/MyDrive/TUS/hashiguchi/data'
os.makedirs(save_dir, exist_ok=True)

def save_esd_results(results, model_name="llama3.2-3b"):
    """
    results を Drive と WandB の両方に最適に保存する関数
    """
    # 辞書を Pandas DataFrame に変換
    df = pd.DataFrame(results)

    # ---------------------------------------------------------
    # ① Google Drive に完全な生データ（固有値含む）を保存 (Pickle形式)
    # ---------------------------------------------------------
    filename = f"{model_name}.pkl"
    filepath = os.path.join(save_dir, filename)

    df.to_pickle(filepath)
    print(f"[Drive] 完全な結果を保存しました: {filepath}")

# 実行例
# results = get_llama_esd_metrics(model)
save_esd_results(result, model_name="llama-3.2-3B")

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from google.colab import drive

# 1. Google Drive のマウント
drive.mount('/content/drive')

# 保存先のディレクトリを作成（ご自身の環境に合わせて変更してください）
save_dir = '/content/drive/MyDrive/TUS/hashiguchi/data'
os.makedirs(save_dir, exist_ok=True)

filename = "llama-3.2-3B.pkl"
filepath = os.path.join(save_dir, filename)

if os.path.exists(filepath):
    result = pd.read_pickle(filepath)
    print(f"✅ データの読み込みが完了しました: {filepath}")
    print(f"行数（層の数）: {len(result)}")
else:
    print(f"❌ ファイルが見つかりません: {filepath}")

In [ ]:
result[result['alpha'] >= 3]

In [ ]:
# 探したい層の名前を指定（LLaMAなどのモデル構造に合わせる）
target_layer_name = "lm_head"

# results['name'] の中から、該当する層のインデックスを探す
try:
    target_index = result['name'].index(target_layer_name)

    # 同じインデックスを使って alpha と alphahat を取得
    layer_alpha = result['alpha'][target_index]
    layer_alphahat = result['alphahat'][target_index]

    print(f"Layer: {target_layer_name}")
    print(f"Alpha: {layer_alpha:.4f}")
    print(f"AlphaHat: {layer_alphahat:.4f}")

except ValueError:
    print(f"エラー: {target_layer_name} が results の中に見つかりませんでした。")

In [ ]:
layer_idx = 2
target_layer = model.model.layers[layer_idx].self_attn.q_proj
# target_layer = model.lm_head

# 重みテンソルを取得し、NumPy配列に変換
W = target_layer.weight.detach().cpu().numpy().astype(np.float32)
print(f"\n[成功] 重み行列を抽出しました (Shape: {W.shape})")
print(f"  - 出力次元 (out_features): {W.shape[0]}")
print(f"  - 入力次元 (in_features): {W.shape[1]}")

# テンソルがPyTorchの場合はNumPyに変換
if hasattr(W, 'detach'):
    W_np = W.detach().cpu().numpy()
else:
    W_np = W

m, n = W_np.shape
gamma = m / n  # LLaMAのq_projなどは 3072x3072 なので gamma=1.0

# 1. 経験的相関行列と固有値の計算
print("固有値を計算中...")
X = W_np @ W_np.T / n
evals = np.linalg.eigvalsh(X)

# 3. 特異値分解 (SVD) とランダム行列理論(RMT)に基づく固有値計算
print("\n特異値分解（SVD）を計算しています...")
# _, S, _ = np.linalg.svd(W, full_matrices=False)

gamma = W.shape[1] / W.shape[0]
m, n = W.shape[1], W.shape[0]
result_orig = funcs1.bema_algorithm1_from_data(W, alpha=0.2, beta=0.1)

print("BEMA Algorithm 1")
print(f"sigma^2_hat = {result_orig['sigma2_hat']:.4f}")
print(f"K_hat        = {result_orig['K_hat']}")
print(f"threshold    = {result_orig['threshold']:.4f}")

sigma2_bema = result_orig['sigma2_hat']
K_bema = result_orig['K_hat']
threshold_bema = result_orig['threshold']
lambda_plus_bema = sigma2_bema * (1 + np.sqrt(gamma))**2
lambda_minus = sigma2_bema * (1 - np.sqrt(gamma))**2
x = np.linspace(lambda_minus, lambda_plus_bema, 500)
rho_mp_bema = np.sqrt((lambda_plus_bema - x) * (x - lambda_minus)) / (2 * np.pi * gamma * x * sigma2_bema)

# 3. プロットの作成
plt.figure(figsize=(14, 6))

# --- 右図：スパイクを含む全体像 ---
plt.hist(evals, bins=150, density=True, color='salmon', alpha=0.7, label='All Eigenvalues')
plt.plot(x, rho_mp_bema, 'b-', lw=2, label='Theoretical MP')
plt.axvline(threshold_bema, color='red', linestyle='--', lw=2, label='Threshold (Signal Start)')

plt.xlim(0, np.max(evals) * 1.05)
# スパイクの密度は非常に低いため、y軸を対数スケールにして見やすくする
plt.yscale('log')
plt.title(f'Full Range: {result_orig["K_hat"]} Spikes (Heavy-Tail)', fontsize=14)
plt.xlabel('Eigenvalue $\lambda$', fontsize=12)
plt.legend()

plt.show()

In [ ]:
layer_idx = 1
target_layer = model.model.layers[layer_idx].self_attn.q_proj

W = target_layer.weight.detach().cpu().numpy().astype(np.float32)

print(f"\n[成功] 重み行列を抽出しました (Shape: {W.shape})")
print(f"  - 出力次元 (out_features): {W.shape[0]}")
print(f"  - 入力次元 (in_features): {W.shape[1]}")

p, n = W.shape

# p <= n にそろえる
if p > n:
    W = W.T
    p, n = W.shape

gamma = min(p, n) / max(p, n)

result_orig = funcs1.bema_algorithm1_from_data(
    W,
    alpha=0.2,
    beta=0.1
)

print("BEMA Algorithm 1")
print(f"sigma^2_hat = {result_orig['sigma2_hat']:.6g}")
print(f"s_hat        = {result_orig['s_hat']}")
print(f"threshold    = {result_orig['threshold']:.6g}")

print("\n特異値分解（SVD）を計算しています...")
S = np.linalg.svd(W, compute_uv=False, full_matrices=False)

# 重要: from_data と同じスケールにする
evals = S ** 2 / n

sigma2_bema = result_orig["sigma2_hat"]
K_bema = result_orig["s_hat"]
threshold_bema = result_orig["threshold"]

max_pn = max(p, n)

lambda_plus_bema = (
    sigma2_bema
    * (1.0 + np.sqrt(gamma)) ** 2
    * (max_pn / n)
)

lambda_minus_bema = (
    sigma2_bema
    * (1.0 - np.sqrt(gamma)) ** 2
    * (max_pn / n)
)

# gamma=1 のとき lambda_minus=0 なので 0 除算を避ける
x_min = max(lambda_minus_bema, 1e-12)
x = np.linspace(x_min, lambda_plus_bema, 500)

# MP density
# rectangular scaling max_pn / n を入れた有効分散
sigma2_eff = sigma2_bema * (max_pn / n)

rho_mp_bema = np.sqrt(
    np.maximum((lambda_plus_bema - x) * (x - lambda_minus_bema), 0.0)
) / (
    2.0 * np.pi * gamma * sigma2_eff * x
)

plt.figure(figsize=(14, 6))

plt.hist(
    evals,
    bins=150,
    density=True,
    color="salmon",
    alpha=0.7,
    label="All Eigenvalues"
)

plt.plot(
    x,
    rho_mp_bema,
    "b-",
    lw=2,
    label="Theoretical MP"
)

plt.axvline(
    threshold_bema,
    color="red",
    linestyle="--",
    lw=2,
    label="BEMA Threshold"
)

plt.xlim(0, np.max(evals) * 1.05)
plt.yscale("log")

# SyntaxWarning 回避
plt.xlabel(r"Eigenvalue $\lambda$", fontsize=12)

plt.legend()
plt.show()

# 全体のmetric分析

In [ ]:
!git clone https://github.com/ryy1210/RMT_utils
import sys
sys.path.append("/content/RMT_utils")

import funcs1

In [ ]:
# transformer block 1つで約4~5分 fix-fingerなら
results = funcs1.get_esd_metrics(model, pl_fitting='fix-finger')

In [ ]:
results = pd.DataFrame(results)
results.columns

In [ ]:
results_numeric = results.select_dtypes(include=np.number)
display(results_numeric.describe())

In [ ]:
results_attn = results[results['name'].str.contains('attn')]
results_attn.describe()

In [ ]:
results_mlp = results[results['name'].str.contains('mlp')]
results_mlp.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 10))
corr_matrix = results_numeric.corr()

# -1から1のスケールで色分けするヒートマップ
sns.heatmap(corr_matrix,
            annot=True,       # セル内に数値を表示
            fmt=".2f",        # 小数点第2位まで表示
            cmap="coolwarm",  # 青から赤へのカラーマップ
            vmin=-1,          # 最小値
            vmax=1,           # 最大値
            linewidths=.5)    # セル間の境界線を設定

plt.title("Correlation Heatmap of results_numeric")
plt.tight_layout()
plt.show()

In [ ]:
plt.scatter(results_attn['alpha'], results_attn['s_hat_ratio_postDE'], label='Attention', color='blue', alpha=0.7)
plt.scatter(results_mlp['alpha'], results_mlp['s_hat_ratio_postDE'], label='MLP', color='orange', alpha=0.7)
plt.xlabel('alpha')
plt.ylabel('s_hat_ratio_postDE')
plt.legend()
plt.show()

In [ ]:
plt.scatter(results_attn['alpha'], results_attn['sigma2_postDE'], label='Attention', color='blue', alpha=0.7)
plt.scatter(results_mlp['alpha'], results_mlp['sigma2_postDE'], label='MLP', color='orange', alpha=0.7)
plt.xlabel('alpha')
plt.ylabel('sigma2_postDE')
plt.plot([1, 2.1], [1, 1], color='red')
plt.legend()
plt.show()

In [ ]:
# Attention層をさらに分割
results_q = results_attn[results_attn['name'].str.contains('q_proj')]
results_k = results_attn[results_attn['name'].str.contains('k_proj')]
results_v = results_attn[results_attn['name'].str.contains('v_proj')]
results_o = results_attn[results_attn['name'].str.contains('o_proj')]

# MLP層をさらに分割
results_gate = results_mlp[results_mlp['name'].str.contains('gate_proj')]
results_up = results_mlp[results_mlp['name'].str.contains('up_proj')]
results_down = results_mlp[results_mlp['name'].str.contains('down_proj')]

print(f"q: {len(results_q)}, k: {len(results_k)}, v: {len(results_v)}, o: {len(results_o)}")
print(f"gate: {len(results_gate)}, up: {len(results_up)}, down: {len(results_down)}")

In [ ]:
# Attention
plt.scatter(results_q['alpha'], results_q['s_hat_ratio_postDE'], label='q_proj', alpha=0.7)
plt.scatter(results_k['alpha'], results_k['s_hat_ratio_postDE'], label='k_proj', alpha=0.7)
plt.scatter(results_v['alpha'], results_v['s_hat_ratio_postDE'], label='v_proj', alpha=0.7)
plt.scatter(results_o['alpha'], results_o['s_hat_ratio_postDE'], label='o_proj', alpha=0.7)

# MLP
plt.scatter(results_gate['alpha'], results_gate['s_hat_ratio_postDE'], label='gate_proj', alpha=0.7)
plt.scatter(results_up['alpha'], results_up['s_hat_ratio_postDE'], label='up_proj', alpha=0.7)
plt.scatter(results_down['alpha'], results_down['s_hat_ratio_postDE'], label='down_proj', alpha=0.7)

plt.xlabel('alpha')
plt.ylabel('s_hat_ratio_postDE')

plt.legend()
plt.show()

In [ ]:
# Attention
plt.scatter(results_q['alpha'], results_q['sigma2_postDE'], label='q_proj', alpha=0.7)
plt.scatter(results_k['alpha'], results_k['sigma2_postDE'], label='k_proj', alpha=0.7)
plt.scatter(results_v['alpha'], results_v['sigma2_postDE'], label='v_proj', alpha=0.7)
plt.scatter(results_o['alpha'], results_o['sigma2_postDE'], label='o_proj', alpha=0.7)

# MLP
plt.scatter(results_gate['alpha'], results_gate['sigma2_postDE'], label='gate_proj', alpha=0.7)
plt.scatter(results_up['alpha'], results_up['sigma2_postDE'], label='up_proj', alpha=0.7)
plt.scatter(results_down['alpha'], results_down['sigma2_postDE'], label='down_proj', alpha=0.7)

plt.plot([1, 2.1], [1, 1], color='red')

plt.xlabel('alpha')
plt.ylabel('sigma2_postDE')

plt.legend()
plt.show()

gateのalphaの値が2群に分かれていた

In [ ]:
plt.hist(results_gate['alpha'],bins=10)

In [ ]:
results_gate[results_gate['alpha'] < 1.6]

In [ ]:
results_q['alpha']

In [ ]:
plt.figure(figsize=(10, 6))

# Attention
plt.plot(results_q['alpha'].values, label='q_proj', marker='o')
plt.plot(results_k['alpha'].values, label='k_proj', marker='o')
plt.plot(results_v['alpha'].values, label='v_proj', marker='o')
plt.plot(results_o['alpha'].values, label='o_proj', marker='o')

# MLP
plt.plot(results_gate['alpha'].values, label='gate_proj', marker='x')
plt.plot(results_up['alpha'].values, label='up_proj', marker='x')
plt.plot(results_down['alpha'].values, label='down_proj', marker='x')

plt.xlabel('Layer Index')
plt.ylabel('alpha')
plt.title('Alpha values across layers for 7 projections')

# 凡例を外側に配置
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import wandb
import pickle
import os
from google.colab import drive

# 1. Google Drive のマウント
drive.mount('/content/drive')

# 保存先のディレクトリを作成（ご自身の環境に合わせて変更してください）
save_dir = '/content/drive/MyDrive/TUS/hashiguchi/data'
os.makedirs(save_dir, exist_ok=True)

def save_esd_results(results, model_name="llama-3.2-3B"):
    """
    results を Drive と WandB の両方に最適に保存する関数
    """
    # 辞書を Pandas DataFrame に変換
    df = pd.DataFrame(results)

    # ---------------------------------------------------------
    # ① Google Drive に完全な生データ（固有値含む）を保存 (Pickle形式)
    # ---------------------------------------------------------
    filename = f"{model_name}_esd_metrics.pkl"
    filepath = os.path.join(save_dir, filename)

    df.to_pickle(filepath)
    print(f"[Drive] 完全な結果を保存しました: {filepath}")

    # ---------------------------------------------------------
    # ② WandB に保存 (Artifacts と Table)
    # ---------------------------------------------------------
    # wandb の run が初期化されていない場合は初期化する
    if wandb.run is None:
        wandb.init(project="LlaMa-RMT-Analysis", name=f"ESD-Metrics-{model_name}")

    # 1. 完全な生データ(.pkl)を Artifact として WandB にバックアップ
    # これにより、後日別の環境からでも `wandb.use_artifact` で重い配列データを復元できます
    artifact = wandb.Artifact(name=f"{model_name}-esd-data", type="dataset")
    artifact.add_file(filepath)
    wandb.log_artifact(artifact)
    print(f"[WandB] 完全な生データを Artifact として保存しました。")

    # 2. WandB ダッシュボードで閲覧するための Table の作成
    # ブラウザの描画負荷を下げるため、数千個の要素を持つ 'eigs' 配列列のみを除外
    if 'eigs' in df.columns:
        df_for_table = df.drop(columns=['eigs'])
    else:
        df_for_table = df.copy()

    wandb_table = wandb.Table(dataframe=df_for_table)
    wandb.log({f"{model_name}_ESD_Metrics": wandb_table})
    print(f"[WandB] メトリクス一覧を Table として保存しました。ダッシュボードで確認できます。")

    # 最後に run を終了する（必要に応じてコメントアウトしてください）
    wandb.finish()


# 実行例
# results = get_esd_metrics(model)
save_esd_results(results, model_name="llama-3.2-3B")

# モデル評価指標

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

def calculate_perplexity(model, tokenizer, device="cuda", max_length=None, stride=512):
    """
    Wikitext-2 データセットを用いてモデルの Perplexity を計算する関数。

    引数:
    - model: ロード済みの Hugging Face モデル
    - tokenizer: ロード済みの トークナイザー
    - device: "cuda" または "cpu"
    - max_length: 一度に処理するトークン長（Noneの場合はモデルの最大長を使用）
    - stride: スライディングウィンドウのストライド幅（処理速度と精度のトレードオフ）
    """
    model.eval()
    model.to(device)

    # 1. Wikitext-2 データセットのテスト用データをロード
    print("Wikitext-2 データセットをロードしています...")
    dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")

    # データセットのテキストをすべて結合
    encodings = tokenizer("\n\n".join(dataset["text"]), return_tensors="pt")

    # 2. シーケンス長の設定
    if max_length is None:
        # Llama 3.2 などの場合は最大長が非常に長いため、メモリ溢れを防ぐために 2048 等に制限することを推奨します
        max_length = min(model.config.max_position_embeddings, 2048)

    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0

    print(f"Perplexityの計算を開始します (総トークン数: {seq_len}, 最大コンテキスト長: {max_length})")

    # 3. スライディングウィンドウで損失 (Negative Log-Likelihood) を計算
    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # 今回新たに予測するトークン数

        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()

        # すでに予測済みのトークン（コンテキスト部分）は損失計算から除外する
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            # outputs.loss はバッチ内の有効なラベル (-100以外) に対する平均クロスエントロピー
            # モデルの損失に予測トークン数を掛けて合計を算出
            neg_log_likelihood = outputs.loss * trg_len

        nlls.append(neg_log_likelihood)

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    # 4. 全トークンにわたる平均負の対数尤度から Perplexity を算出
    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)

    return ppl.item()

In [ ]:
# ==========================================
# 実行例
# ==========================================
# ※すでに手元で Llama-3.2-3B などをロードしている場合は、この部分はスキップして
#   ご自身の model と tokenizer をそのまま関数に渡してください。

model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

# PPLを計算
ppl_score = calculate_perplexity(model, tokenizer, device="cuda", max_length=2048, stride=512)
print(f"\n✅ 計算完了: Wikitext-2 Perplexity = {ppl_score:.4f}")

# LRA

## 一層ずつLRAしていく

### alpha大きい順（仮）

In [ ]:
import random
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 1. Alphaが大きい順（ノイズが多く、圧縮しても影響が少ない層）に並べる
lra_list_alpha = results.sort_values(by='alpha', ascending=False)['name'].tolist()

MAX_LAYERS = 20

# --- 実験1: Alpha順 ---
print("=== Alpha順での実験を開始 ===")
# model = get_model_from_huggingface(...) # モデルを初期化(ロード)
history_alpha = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    DE=True,
    seq_len = 1024,
    batch_size = 2
)

plt.plot(history_alpha['step'], history_alpha['ppl'], label='Alpha-based (Ascending)')

### ランダムな順番にLRA（iter = 10)

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 事前に results_df, tokenizer, run_lra_experiment が定義・準備されている前提

# 1. Alphaが大きい順（ノイズが多く、圧縮しても影響が少ない層）に並べる
lra_list_alpha = results.sort_values(by='alpha', ascending=False)['name'].tolist()

MAX_LAYERS = 20  # 例として最初の20層で比較
iter_count = 10
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"

# ==========================================
# 実験: ランダム順での LRA (iter_count 回繰り返す)
# ==========================================
for i in range(iter_count):
    print(f"\n{'='*20}\n===== ランダム実験 {i+1}/{iter_count} ======\n{'='*20}")

    # 2. ランダムな順序のリストを作る
    lra_list_random = lra_list_alpha.copy()
    random.shuffle(lra_list_random)

    # 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
    run_name = f"Random_Order_iter{i+1}"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": "Random",
            "iteration": i + 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True # ループ内で連続してinitを呼ぶために必要
    )

    # 4. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="cpu"
    )

    # 5. 実験を実行
    # （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results,
        lra_list=lra_list_random,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len = 1024,
        batch_size = 2
    )

    # 6. 結果を WandB にログとして送信
    # history_random (DataFrame) の行を1つずつ WandB に記録する
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],                # 圧縮した層の数 (0はベースライン)
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
            "alpha_val": row['alpha_of_layer']
        })

    # 7. Matplotlib でもローカルに描画 (オプション)
    plt.plot(history_random['step'], history_random['ppl'], alpha=0.5, label=f'Random {i+1}')

    # ==========================================
    # 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    # 1. モデル変数を削除
    del model

    # 3. PyTorch の GPU キャッシュを完全にクリア
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

    print(f"🧹 イテレーション {i+1} 終了: GPUメモリを解放しました。")
    # ==========================================
    wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: Random Order")
plt.show()

### alphaが大きい順

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 事前に results_df, tokenizer, run_lra_experiment が定義・準備されている前提

# 1. Alphaが大きい順（ノイズが多く、圧縮しても影響が少ない層）に並べる
lra_list_alpha = results.sort_values(by='alpha', ascending=False)['name'].tolist()

MAX_LAYERS = 20  # 例として最初の20層で比較
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"alpha_descending"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "alpha_descending",
        "iteration": 1,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": True
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=True,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: descending Order")
plt.show()

### alphaが小さい順

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 事前に results_df, tokenizer, run_lra_experiment が定義・準備されている前提

# 1. Alphaが小さい順に並べる
lra_list_alpha = results.sort_values(by='alpha', ascending=True)['name'].tolist()

MAX_LAYERS = 20  # 例として最初の20層で比較
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"alpha_ascending"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "alpha_ascending",
        "iteration": 1,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": True
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=True,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: ascending Order")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 事前に results_df, tokenizer, run_lra_experiment が定義・準備されている前提

# 1. Alphaが小さい順に並べる
lra_list_alpha = results.sort_values(by='alpha', ascending=True)['name'].tolist()

MAX_LAYERS = 100  # 100層までLRAしていく
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"alpha_ascending_2"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "alpha_ascending",
        "iteration": 2,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": True
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=True,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: ascending Order")
plt.show()

### KS_postDE_1ベースの順番でのLRA

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 事前に results_df, tokenizer, run_lra_experiment が定義・準備されている前提

# 1.
lra_list_alpha = results.sort_values(by='KS_postDE_1', ascending=False)['name'].tolist()

MAX_LAYERS = 20
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"KS_postDE_1_descending"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "KS_postDE_1_descending",
        "iteration": 1,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": True
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=True,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: KS_postDE_1 descending Order")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 1. Alphaが小さい順に並べる
lra_list_alpha = results.sort_values(by='KS_postDE_1', ascending=False)['name'].tolist()

MAX_LAYERS = 100
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"KS_postDE_1_descending_2"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "KS_postDE_1_descending",
        "iteration": 2,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": True
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=True,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: KS_postDE_1 descending Order")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 事前に results_df, tokenizer, run_lra_experiment が定義・準備されている前提

# 1. Alphaが小さい順に並べる
lra_list_alpha = results.sort_values(by='KS_postDE_1', ascending=True)['name'].tolist()

MAX_LAYERS = 20
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"KS_postDE_1_ascending"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "KS_postDE_1_ascending",
        "iteration": 1,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": True
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=True,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: KS_postDE_1 ascending Order")
plt.show()

### s_hat_ratio

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 1. s_hat_ratio_postDEが大きい順に並べる
lra_list_s = results.sort_values(by='s_hat_ratio_postDE', ascending=False)['name'].tolist()

MAX_LAYERS = 100
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"s_hat_ratio_postDE_descending"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "s_hat_ratio_postDE_descending",
        "iteration": 1,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": True
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_s,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=True,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: s_hat_ratio_postDE descending Order")
plt.show()

### モジュールごとに

#### alpha descending

In [ ]:
lra_list_alpha = results.sort_values(by='alpha', ascending=False)['name'].tolist()

# 7つのリストを初期化
list_q = []
list_k = []
list_v = []
list_o = []
list_gate = []
list_up = []
list_down = []

# lra_list_alpha はすでに alpha の降順にソートされているため、
# 順番に各リストに振り分けるだけで条件を満たせます
for name in lra_list_alpha:
    if 'q_proj' in name:
        list_q.append(name)
    elif 'k_proj' in name:
        list_k.append(name)
    elif 'v_proj' in name:
        list_v.append(name)
    elif 'o_proj' in name:
        list_o.append(name)
    elif 'gate_proj' in name:
        list_gate.append(name)
    elif 'up_proj' in name:
        list_up.append(name)
    elif 'down_proj' in name:
        list_down.append(name)

list_alpha = [list_q,list_k,list_v,list_o,list_gate,list_up,list_down]

# 確認
print(f"Q_proj: {len(list_q)}")
print(f"K_proj: {len(list_k)}")
print(f"V_proj: {len(list_v)}")
print(f"O_proj: {len(list_o)}")
print(f"Gate_proj: {len(list_gate)}")
print(f"Up_proj: {len(list_up)}")
print(f"Down_proj: {len(list_down)}")

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 事前に results_df, tokenizer, run_lra_experiment が定義・準備されている前提

MAX_LAYERS = 28  # 各モジュールで全部
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"

module_name = ['q', 'k', 'v', 'o', 'gate', 'up' , 'down']

for i in range(7):

    lra_list = list_alpha[i]
    f = module_name[i]

    print(f"\n{'='*20}\n===== {f} ======\n{'='*20}")

    # 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
    run_name = f"module wise {f} alpha descending"
    wandb.init(
        project=wandb_project_name,
        name=run_name,
        config={
            "method": f"module wise {f} alpha descending",
            "iteration": 1,
            "max_layers": MAX_LAYERS,
            "model_id": model_id,
            "DE": True
        },
        reinit=True # ループ内で連続してinitを呼ぶために必要
    )

    # 4. クリーンなモデルをロード (毎回初期化)
    print(f"{model_id} を読み込んでいます...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="cpu"
    )

    # 5. 実験を実行
    # （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
    history_random = funcs1.run_lra_experiment(
        model, tokenizer, results,
        lra_list=lra_list,
        max_lra_layers=MAX_LAYERS,
        dataset_name='wikitext2',
        DE=True,
        seq_len = 1024,
        batch_size = 2
    )

    # 6. 結果を WandB にログとして送信
    # history_random (DataFrame) の行を1つずつ WandB に記録する
    for index, row in history_random.iterrows():
        wandb.log({
            "step": row['step'],                # 圧縮した層の数 (0はベースライン)
            "layer_name": row['layer_compressed'],
            "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
            "alpha_val": row['alpha_of_layer']
        })

    # 7. Matplotlib でもローカルに描画 (オプション)
    plt.plot(history_random['step'], history_random['ppl'], alpha=0.5, label=f'{f}')

    # ==========================================
    # 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
    # ==========================================
    # 1. モデル変数を削除
    del model

    # 3. PyTorch の GPU キャッシュを完全にクリア
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

    print(f"🧹 イテレーション終了: GPUメモリを解放しました。")
    # ==========================================
    wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: module wise alpha descending")
plt.show()

## DEのOn/Offの比較

alphaが小さい順, KSが大きい順でやったときのDEの有無を比較する
DE Onは上でやったものそのまま
DE Offをここで

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 事前に results_df, tokenizer, run_lra_experiment が定義・準備されている前提

# 1. Alphaが小さい順に並べる
lra_list_alpha = results.sort_values(by='alpha', ascending=True)['name'].tolist()

MAX_LAYERS = 100  # 100層までLRAしていく
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"alpha_ascending_OffDE"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "alpha_ascending",
        "iteration": 1,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": False
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=False,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: ascending Order")
plt.show()

In [ ]:
import wandb
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


# 1. Alphaが小さい順に並べる
lra_list_alpha = results.sort_values(by='KS_postDE_1', ascending=False)['name'].tolist()

MAX_LAYERS = 100
model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# --- [追加] WandB の初期化設定 ---
# プロジェクト名は適宜変更してください
wandb_project_name = "LRA-Ablation-Study"


# 3. WandB の Run を初期化 (イテレーションごとに新しいRunを作成)
run_name = f"KS_postDE_1_descending_OffDE"
wandb.init(
    project=wandb_project_name,
    name=run_name,
    config={
        "method": "KS_postDE_1_descending",
        "iteration": 1,
        "max_layers": MAX_LAYERS,
        "model_id": model_id,
        "DE": False
    },
    reinit=True # ループ内で連続してinitを呼ぶために必要
)

# 4. クリーンなモデルをロード (毎回初期化)
print(f"{model_id} を読み込んでいます...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)

# 5. 実験を実行
# （※ run_lra_experiment 内で get_ppl が呼ばれ、リストごとの PPL が得られる）
history_random = funcs1.run_lra_experiment(
    model, tokenizer, results,
    lra_list=lra_list_alpha,
    max_lra_layers=MAX_LAYERS,
    dataset_name='wikitext2',
    DE=False,
    seq_len = 1024,
    batch_size = 2
)

# 6. 結果を WandB にログとして送信
# history_random (DataFrame) の行を1つずつ WandB に記録する
for index, row in history_random.iterrows():
    wandb.log({
        "step": row['step'],                # 圧縮した層の数 (0はベースライン)
        "layer_name": row['layer_compressed'],
        "ppl_wikitext2": row['ppl'],        # これがグラフのY軸になります
        "alpha_val": row['alpha_of_layer']
    })

# 7. Matplotlib でもローカルに描画 (オプション)
plt.plot(history_random['step'], history_random['ppl'], alpha=0.5)

# ==========================================
# 🚨 【超重要】OOMを防ぐためのメモリ完全解放処理
# ==========================================
# 1. モデル変数を削除
del model

# 3. PyTorch の GPU キャッシュを完全にクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # IPCメモリの回収 (オプション)

# ==========================================
wandb.finish() # 現在のRunを終了して次のイテレーションへ

# Matplotlib の仕上げ
plt.xlabel("Number of Compressed Layers")
plt.ylabel("Wikitext-2 PPL")
plt.title("LRA Perplexity Degradation: KS_postDE_1 descending Order")
plt.show()

In [ ]:
import wandb
import pandas as pd
import torch.nn as nn
import numpy as np

def update_specific_runs_with_reduction(model, results_df, entity_project_path, target_run_names, new_project_name, DE=True):
    """
    指定した特定のRunデータを取得し、パラメータ削減率を追加して新しいプロジェクトにアップロードする。
    """
    api = wandb.Api()

    print(f"プロジェクト {entity_project_path} からRunを取得中...")
    runs = api.runs(entity_project_path)

    # モデルの形状をキャッシュ（検索の高速化）
    layer_shapes = {}
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            layer_shapes[name] = module.weight.shape

    total_original_params = sum(p.numel() for p in model.parameters())
    results_indexed = results_df.set_index('name')

    # 対象のRunを見つけて処理
    for old_run in runs:
        if old_run.name not in target_run_names:
            continue # 対象リストにないRunは無視

        print(f"\n🔄 ターゲットRunを処理中: {old_run.name}")

        # 過去のデータをダウンロード
        history_df = old_run.history()

        if 'layer_name' not in history_df.columns:
            print("  -> [エラー] layer_name カラムが存在しません。スキップします。")
            continue

        # ステップ順にソートしてインデックスをリセット
        history_df = history_df.sort_values(by='step').reset_index(drop=True)

        current_total_params = total_original_params
        new_logs = []

        # 削減率の計算ループ
        for idx, row in history_df.iterrows():
            step = row['step']
            layer_name = row['layer_name']

            # Step 0 (ベースライン) または不明な層の処理
            if layer_name == "baseline" or pd.isna(layer_name) or layer_name not in layer_shapes or layer_name not in results_indexed.index:
                ratio = (1.0 - current_total_params / total_original_params) * 100
            else:
                # 元のパラメータ数の計算
                shape = layer_shapes[layer_name]
                m_orig, n_orig = shape[0], shape[1]
                original_layer_params = m_orig * n_orig

                # 圧縮後のパラメータ数の計算
                # DE=False のRunの場合は s_hat_preDE を使うなどの判定を入れる
                s_col = 's_hat_postDE' if DE else 's_hat_preDE'

                # Runの名前に "OffDE" が含まれている場合は強制的に preDE (DE=False) を使用する
                if "OffDE" in old_run.name:
                    s_col = 's_hat_preDE'

                s_hat = int(results_indexed.loc[layer_name, s_col])
                compressed_layer_params = s_hat * (m_orig + n_orig)

                # 削減量の更新
                params_saved = original_layer_params - compressed_layer_params
                if params_saved > 0:
                    current_total_params -= params_saved

                ratio = (1.0 - current_total_params / total_original_params) * 100

            # ロギング用データの作成
            new_log_entry = {
                "step": step,
                "layer_name": layer_name,
                "ppl_wikitext2": row['ppl_wikitext2'],
                "reduction_ratio_percent": ratio # ✨ 新しく追加した指標
            }

            # もし元のデータに alpha_val があればそれも引き継ぐ
            if 'alpha_val' in row:
                new_log_entry["alpha_val"] = row['alpha_val']

            new_logs.append(new_log_entry)

        # 3. 新しい Run を立ち上げてデータを送信
        new_run_name = f"{old_run.name}_updated"
        print(f"  -> 新しいRun '{new_run_name}' としてプロジェクト '{new_project_name}' にアップロードします...")

        wandb.init(
            project=new_project_name,
            name=new_run_name,
            config=old_run.config, # 古い設定を引き継ぐ
            reinit=True
        )

        for log_entry in new_logs:
            wandb.log(log_entry)

        wandb.finish()

    print("\n✅ 指定されたすべてのRunのアップデートが完了しました！")

# ==========================================
# 実行部分
# ==========================================
# 対象のRunの名前のリスト
# target_runs = [
#     "s_hat_ratio_postDE_descending_OffDE",
#     "KS_postDE_1_descending_OffDE",
#     "s_hat_ratio_postDE_descending"
# ]
target_runs = ["alpha_ascending_OffDE"]


# 自分のWandBのユーザー名（またはエンティティ名）に書き換えてください
USER_NAME = "ryoya-zushi1210-tokyo-university-of-science"
OLD_PROJECT_PATH = f"{USER_NAME}/LRA-Ablation-Study"
NEW_PROJECT_NAME = "LRA-Ablation-Study-Updated"

# 実行
update_specific_runs_with_reduction(
    model=model,
    results_df=results,
    entity_project_path=OLD_PROJECT_PATH,
    target_run_names=target_runs,
    new_project_name=NEW_PROJECT_NAME,
    DE=True # 基本はTrueで、関数内でRun名にOffDEがあればFalseに切り替えるようにしています
)

## 削減したパラメータ数計算

In [ ]:
import wandb
import pandas as pd
import torch.nn as nn

def relog_wandb_with_reductions(model, results_df, entity_project_path, DE=True):
    """
    WandB上の既存のRunデータをダウンロードし、パラメータ削減率を計算して
    新しいRunとして記録し直すスクリプト。

    Parameters:
        model: 形状情報を取得するためのPyTorchモデル (GPU不要)
        results_df: s_hat情報を持つDataFrame
        entity_project_path: "ユーザー名/プロジェクト名" (例: "ryoya/LRA-Ablation-Study")
    """
    api = wandb.Api()

    # 1. 対象プロジェクトのすべてのRunを取得
    print(f"プロジェクト {entity_project_path} からRunを取得中...")
    runs = api.runs(entity_project_path)

    # モデルの形状をキャッシュ（検索の高速化）
    layer_shapes = {}
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            layer_shapes[name] = module.weight.shape

    total_original_params = sum(p.numel() for p in model.parameters())
    results_indexed = results_df.set_index('name')

    # 2. 各Runについて処理を行う
    for old_run in runs:
        # すでに "reduction_ratio_percent" があるRunはスキップ（二重処理防止）
        if "reduction_ratio_percent" in old_run.summary:
            continue

        print(f"\n🔄 Runを再処理中: {old_run.name}")

        # 過去のデータをダウンロード (数秒で終わります)
        history_df = old_run.history()

        # もし history に layer_name が保存されていなければスキップ
        if 'layer_name' not in history_df.columns:
            print("  -> layer_name カラムがないためスキップしました。")
            continue

        # --- パラメータ削減率の計算 ---
        current_total_params = total_original_params
        new_logs = []

        # ステップ順にソート（念のため）
        history_df = history_df.sort_values(by='step').reset_index(drop=True)

        for idx, row in history_df.iterrows():
            step = row['step']
            layer_name = row['layer_name']

            # Step 0 (ベースライン) または不明な層の処理
            if layer_name == "baseline" or layer_name not in layer_shapes or layer_name not in results_indexed.index:
                ratio = (1.0 - current_total_params / total_original_params) * 100
            else:
                # 削減量の計算
                shape = layer_shapes[layer_name]
                m_orig, n_orig = shape[0], shape[1]
                original_layer_params = m_orig * n_orig

                s_col = 's_hat_postDE' if DE else 's_hat_preDE'
                s_hat = int(results_indexed.loc[layer_name, s_col])
                compressed_layer_params = s_hat * (m_orig + n_orig)

                params_saved = original_layer_params - compressed_layer_params
                if params_saved > 0:
                    current_total_params -= params_saved

                ratio = (1.0 - current_total_params / total_original_params) * 100

            # ロギング用にデータをまとめる
            new_log_entry = {
                "step": step,
                "layer_name": layer_name,
                "ppl_wikitext2": row['ppl_wikitext2'],
                "alpha_val": row.get('alpha_val', float('nan')), # 旧データに存在すれば取得
                "reduction_ratio_percent": ratio                 # ✨ 新しく追加した指標
            }
            new_logs.append(new_log_entry)

        # 3. 新しい Run を立ち上げてデータを送信
        new_run_name = f"{old_run.name}_with_params"
        print(f"  -> 新しいRun '{new_run_name}' として WandB にアップロードします...")

        # 新しいプロジェクト名にしてもOKです（混ざるのが嫌な場合）
        new_project_name = entity_project_path.split('/')[1] + "-Updated"

        wandb.init(
            project=new_project_name,
            name=new_run_name,
            config=old_run.config, # 古い設定も引き継ぐ
            reinit=True
        )

        for log_entry in new_logs:
            wandb.log(log_entry)

        wandb.finish()

    print("\n✅ すべての過去データのアップデートが完了しました！")

# 実行例
relog_wandb_with_reductions(model, results, "ryoya-zushi1210-tokyo-university-of-science/LRA-Ablation-Study", DE=True)

## alpha_threshold = 1.4

In [ ]:
# fast_SVD = False で30分くらい
model_lra, compressed_layers, param_dict = funcs1.apply_lra(
    model=model,
    results=results,
    alpha_threshold=1.4,
    DE=True,
    fast_SVD=True
)

In [ ]:
import torch
import torch.nn as nn

def print_compression_stats(model, param_dict):
    """
    元のモデルと圧縮後の実効パラメータ数 (param_dict) を比較し、削減率を計算して表示する
    """

    # 1. モデル全体の元のパラメータ数を計算
    total_original_params = sum(p.numel() for p in model.parameters())

    # 2. Linear層の元のパラメータ数を計算
    original_linear_params = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            original_linear_params += module.weight.numel()
            if module.bias is not None:
                original_linear_params += module.bias.numel()

    # 3. Linear層の圧縮後（実効）パラメータ数を集計
    compressed_linear_params = sum(param_dict.values())

    # 4. モデル全体の圧縮後パラメータ数を算出
    # (元の全体数から、Linear層の元の数を引き、圧縮後のLinear層の数を足す)
    total_compressed_params = total_original_params - original_linear_params + compressed_linear_params

    # --- 結果の表示 ---
    print("="*50)
    print("📉 パラメータ削減レポート")
    print("="*50)

    # Linear層のみの比較
    lin_reduction = (1 - compressed_linear_params / original_linear_params) * 100
    print("[Linear層単体の比較]")
    print(f"  元のパラメータ数 : {original_linear_params / 1e6:.2f} M (百万)")
    print(f"  圧縮後パラメータ数 : {compressed_linear_params / 1e6:.2f} M (百万)")
    print(f"  ✨ 削減率         : {lin_reduction:.2f} % 削減")
    print("-" * 50)

    # モデル全体の比較 (EmbeddingやLayerNormなどを含む)
    total_reduction = (1 - total_compressed_params / total_original_params) * 100
    print("[モデル全体の比較]")
    print(f"  元のパラメータ数 : {total_original_params / 1e6:.2f} M (百万)")
    print(f"  圧縮後パラメータ数 : {total_compressed_params / 1e6:.2f} M (百万)")
    print(f"  ✨ 削減率         : {total_reduction:.2f} % 削減")
    print("="*50)


print_compression_stats(model, param_dict)

In [ ]:
def check_model_params(model):
    bad_params = []

    for name, p in model.named_parameters():
        if p is None:
            continue

        if not torch.isfinite(p).all():
            num_nan = torch.isnan(p).sum().item()
            num_inf = torch.isinf(p).sum().item()
            bad_params.append((name, num_nan, num_inf, p.dtype, tuple(p.shape)))

    if len(bad_params) == 0:
        print("✅ 全パラメータは finite です")
    else:
        print("❌ NaN / Inf を含むパラメータがあります")
        for item in bad_params:
            print(item)

    return bad_params

In [ ]:
bad_params = check_model_params(model_lra)

In [ ]:
model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)


# PPLを計算
ppl_score = calculate_perplexity(model_lra, tokenizer, device="cuda", max_length=2048, stride=512)
print(f"\n✅ 計算完了: Wikitext-2 Perplexity = {ppl_score:.4f}")

## alpha_threshold = 1.7

In [ ]:
# fast_SVD = True で25分くらい
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu" # 念のため一度CPUに展開
)
model_lra2, compressed_layers2, param_dict2 = funcs1.apply_lra(
    model=model,
    results=results,
    alpha_threshold=1.7,
    DE=True,
    fast_SVD=True
)

In [ ]:
import torch
import torch.nn as nn

def print_compression_stats(model, param_dict):
    """
    元のモデルと圧縮後の実効パラメータ数 (param_dict) を比較し、削減率を計算して表示する
    """

    # 1. モデル全体の元のパラメータ数を計算
    total_original_params = sum(p.numel() for p in model.parameters())

    # 2. Linear層の元のパラメータ数を計算
    original_linear_params = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            original_linear_params += module.weight.numel()
            if module.bias is not None:
                original_linear_params += module.bias.numel()

    # 3. Linear層の圧縮後（実効）パラメータ数を集計
    compressed_linear_params = sum(param_dict.values())

    # 4. モデル全体の圧縮後パラメータ数を算出
    # (元の全体数から、Linear層の元の数を引き、圧縮後のLinear層の数を足す)
    total_compressed_params = total_original_params - original_linear_params + compressed_linear_params

    # --- 結果の表示 ---
    print("="*50)
    print("📉 パラメータ削減レポート")
    print("="*50)

    # Linear層のみの比較
    lin_reduction = (1 - compressed_linear_params / original_linear_params) * 100
    print("[Linear層単体の比較]")
    print(f"  元のパラメータ数 : {original_linear_params / 1e6:.2f} M (百万)")
    print(f"  圧縮後パラメータ数 : {compressed_linear_params / 1e6:.2f} M (百万)")
    print(f"  ✨ 削減率         : {lin_reduction:.2f} % 削減")
    print("-" * 50)

    # モデル全体の比較 (EmbeddingやLayerNormなどを含む)
    total_reduction = (1 - total_compressed_params / total_original_params) * 100
    print("[モデル全体の比較]")
    print(f"  元のパラメータ数 : {total_original_params / 1e6:.2f} M (百万)")
    print(f"  圧縮後パラメータ数 : {total_compressed_params / 1e6:.2f} M (百万)")
    print(f"  ✨ 削減率         : {total_reduction:.2f} % 削減")
    print("="*50)


print_compression_stats(model, param_dict2)

In [ ]:
def check_model_params(model):
    bad_params = []

    for name, p in model.named_parameters():
        if p is None:
            continue

        if not torch.isfinite(p).all():
            num_nan = torch.isnan(p).sum().item()
            num_inf = torch.isinf(p).sum().item()
            bad_params.append((name, num_nan, num_inf, p.dtype, tuple(p.shape)))

    if len(bad_params) == 0:
        print("✅ 全パラメータは finite です")
    else:
        print("❌ NaN / Inf を含むパラメータがあります")
        for item in bad_params:
            print(item)

    return bad_params

In [ ]:
bad_params = check_model_params(model_lra2)

In [ ]:
del model_lra

In [ ]:
model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)


# PPLを計算
ppl_score = calculate_perplexity(model_lra2, tokenizer, device="cuda", max_length=2048, stride=512)
print(f"\n✅ 計算完了: Wikitext-2 Perplexity = {ppl_score:.4f}")

# alpha pruning

In [ ]:
import gc
import wandb
import random
import matplotlib.pyplot as plt
import torch
import pandas as pd

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


from funcs1 import get_ppl

model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

wandb_project_name = "Alpha-Pruning-Ablation-Study"

sparsity_list = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7]

seq_len = 1024
batch_size = 2
dataset_name = "wikitext2"

print("【Baseline】枝刈りなしモデルの PPL を計算します")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
)

model.eval()

baseline_ppl = get_ppl(
    model,
    tokenizer,
    dataset_name=dataset_name,
    seq_len=seq_len,
    batch_size=batch_size,
)

del model
gc.collect()
torch.cuda.empty_cache()

experiments = [
    {
        "run_name": "alpha_forward_magnitude",
        "alpha_prune": True,
        "alpha_reverse": False,
        "prune_metric": "magnitude",
        "blockwise": False,
    },
    {
        "run_name": "alpha_forward_magnitude_blockwise",
        "alpha_prune": True,
        "alpha_reverse": False,
        "prune_metric": "magnitude",
        "blockwise": True,
    },
    {
        "run_name": "alpha_reverse_magnitude",
        "alpha_prune": True,
        "alpha_reverse": True,
        "prune_metric": "magnitude",
        "blockwise": False,
    },
    {
        "run_name": "alpha_reverse_magnitude_blockwise",
        "alpha_prune": True,
        "alpha_reverse": True,
        "prune_metric": "magnitude",
        "blockwise": True,
    },
    {
        "run_name": "uniform_magnitude",
        "alpha_prune": False,
        "alpha_reverse": False,
        "prune_metric": "magnitude",
        "blockwise": False,
    },
]


all_histories = {}

for exp in experiments:

    wandb.init(
        project=wandb_project_name,
        name=exp["run_name"],
        config={
            "method": "alpha_pruning" if exp["alpha_prune"] else "uniform_pruning",
            "alpha_reverse": exp["alpha_reverse"],
            "prune_metric": exp["prune_metric"],
            "blockwise": exp["blockwise"],
            "sparsity_list": sparsity_list,
            "model_id": model_id,
            "dataset_name": dataset_name,
            "seq_len": seq_len,
            "batch_size": batch_size,
        },
        reinit=True,
    )

    history = []

    # ------------------------------------------------------------
    # step 0: baseline を各 run に記録
    # ------------------------------------------------------------
    baseline_row = {
        "step": 0,
        "target_sparsity": 0.0,
        "actual_sparsity": 0.0,
        "ppl": baseline_ppl,
        "alpha_prune": exp["alpha_prune"],
        "alpha_reverse": exp["alpha_reverse"],
        "prune_metric": exp["prune_metric"],
        "blockwise": exp["blockwise"],
    }

    history.append(baseline_row)

    wandb.log({
        "step": 0,
        "target_sparsity": 0.0,
        "actual_sparsity": 0.0,
        "ppl_wikitext2": baseline_ppl,
        "alpha_prune": exp["alpha_prune"],
        "alpha_reverse": exp["alpha_reverse"],
        "prune_metric": exp["prune_metric"],
        "blockwise": exp["blockwise"],
    })

    # ------------------------------------------------------------
    # 各 sparsity で fresh model をロードして実験
    # ------------------------------------------------------------
    for step, sp in enumerate(sparsity_list, start=1):

        print("\n" + "=" * 80)
        print(f'Run: {exp["run_name"]} | Step {step}/{len(sparsity_list)} | sparsity={sp}')
        print("=" * 80)

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto",
        )

        model.eval()

        result = funcs1.run_pruning_experiment(
            model=model,
            tokenizer=tokenizer,
            results_df=results,
            dataset_name=dataset_name,
            seq_len=seq_len,
            batch_size=batch_size,
            sparsity=sp,   # list ではなく float を渡す
            alpha_prune=exp["alpha_prune"],
            prune_metric=exp["prune_metric"],
            blockwise=exp["blockwise"],
            alpha_reverse=exp["alpha_reverse"],
        )

        result["step"] = step

        history.append(result)

        wandb.log({
            "step": result["step"],
            "target_sparsity": result["target_sparsity"],
            "actual_sparsity": result["actual_sparsity"],
            "ppl_wikitext2": result["ppl"],
            "alpha_prune": result["alpha_prune"],
            "alpha_reverse": result["alpha_reverse"],
            "prune_metric": result["prune_metric"],
            "blockwise": result["blockwise"],
        })

        del model
        gc.collect()
        torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # DataFrame 化して WandB Table に保存
    # ------------------------------------------------------------
    history_df = pd.DataFrame(history)

    wandb.log({
        "history_table": wandb.Table(dataframe=history_df)
    })

    all_histories[exp["run_name"]] = history_df

    wandb.finish()

    plt.figure(figsize=(8, 5))

for run_name, history_df in all_histories.items():
    plt.plot(
        history_df["target_sparsity"],
        history_df["ppl"],
        marker="o",
        alpha=0.7,
        label=run_name,
    )

plt.xlabel("Target sparsity")
plt.ylabel("PPL")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import gc
import wandb
import random
import matplotlib.pyplot as plt
import torch
import pandas as pd

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm


from funcs1 import get_ppl

model_id = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

wandb_project_name = "Alpha-Pruning-Ablation-Study"

sparsity_list = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7]

seq_len = 1024
batch_size = 2
dataset_name = "wikitext2"

print("【Baseline】枝刈りなしモデルの PPL を計算します")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
)

model.eval()

baseline_ppl = get_ppl(
    model,
    tokenizer,
    dataset_name=dataset_name,
    seq_len=seq_len,
    batch_size=batch_size,
)

del model
gc.collect()
torch.cuda.empty_cache()

experiments = [
    {
        "run_name": "alpha_forward_random",
        "alpha_prune": True,
        "alpha_reverse": False,
        "prune_metric": "random",
        "blockwise": False,
    },
    {
        "run_name": "alpha_forward_random_blockwise",
        "alpha_prune": True,
        "alpha_reverse": False,
        "prune_metric": "random",
        "blockwise": True,
    },
    {
        "run_name": "alpha_reverse_random",
        "alpha_prune": True,
        "alpha_reverse": True,
        "prune_metric": "random",
        "blockwise": False,
    },
    {
        "run_name": "alpha_reverse_random_blockwise",
        "alpha_prune": True,
        "alpha_reverse": True,
        "prune_metric": "random",
        "blockwise": True,
    },
    {
        "run_name": "uniform_random",
        "alpha_prune": False,
        "alpha_reverse": False,
        "prune_metric": "random",
        "blockwise": False,
    },
]


all_histories = {}

for exp in experiments:

    wandb.init(
        project=wandb_project_name,
        name=exp["run_name"],
        config={
            "method": "alpha_pruning" if exp["alpha_prune"] else "uniform_pruning",
            "alpha_reverse": exp["alpha_reverse"],
            "prune_metric": exp["prune_metric"],
            "blockwise": exp["blockwise"],
            "sparsity_list": sparsity_list,
            "model_id": model_id,
            "dataset_name": dataset_name,
            "seq_len": seq_len,
            "batch_size": batch_size,
        },
        reinit=True,
    )

    history = []

    # ------------------------------------------------------------
    # step 0: baseline を各 run に記録
    # ------------------------------------------------------------
    baseline_row = {
        "step": 0,
        "target_sparsity": 0.0,
        "actual_sparsity": 0.0,
        "ppl": baseline_ppl,
        "alpha_prune": exp["alpha_prune"],
        "alpha_reverse": exp["alpha_reverse"],
        "prune_metric": exp["prune_metric"],
        "blockwise": exp["blockwise"],
    }

    history.append(baseline_row)

    wandb.log({
        "step": 0,
        "target_sparsity": 0.0,
        "actual_sparsity": 0.0,
        "ppl_wikitext2": baseline_ppl,
        "alpha_prune": exp["alpha_prune"],
        "alpha_reverse": exp["alpha_reverse"],
        "prune_metric": exp["prune_metric"],
        "blockwise": exp["blockwise"],
    })

    # ------------------------------------------------------------
    # 各 sparsity で fresh model をロードして実験
    # ------------------------------------------------------------
    for step, sp in enumerate(sparsity_list, start=1):

        print("\n" + "=" * 80)
        print(f'Run: {exp["run_name"]} | Step {step}/{len(sparsity_list)} | sparsity={sp}')
        print("=" * 80)

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto",
        )

        model.eval()

        result = funcs1.run_pruning_experiment(
            model=model,
            tokenizer=tokenizer,
            results_df=results,
            dataset_name=dataset_name,
            seq_len=seq_len,
            batch_size=batch_size,
            sparsity=sp,   # list ではなく float を渡す
            alpha_prune=exp["alpha_prune"],
            prune_metric=exp["prune_metric"],
            blockwise=exp["blockwise"],
            alpha_reverse=exp["alpha_reverse"],
        )

        result["step"] = step

        history.append(result)

        wandb.log({
            "step": result["step"],
            "target_sparsity": result["target_sparsity"],
            "actual_sparsity": result["actual_sparsity"],
            "ppl_wikitext2": result["ppl"],
            "alpha_prune": result["alpha_prune"],
            "alpha_reverse": result["alpha_reverse"],
            "prune_metric": result["prune_metric"],
            "blockwise": result["blockwise"],
        })

        del model
        gc.collect()
        torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # DataFrame 化して WandB Table に保存
    # ------------------------------------------------------------
    history_df = pd.DataFrame(history)

    wandb.log({
        "history_table": wandb.Table(dataframe=history_df)
    })

    all_histories[exp["run_name"]] = history_df

    wandb.finish()

    plt.figure(figsize=(8, 5))

for run_name, history_df in all_histories.items():
    plt.plot(
        history_df["target_sparsity"],
        history_df["ppl"],
        marker="o",
        alpha=0.7,
        label=run_name,
    )

plt.xlabel("Target sparsity")
plt.ylabel("PPL")
plt.legend()
plt.grid(True)
plt.show()

## alpha vs applied sparsity

### Target Sparsity: 20.0%, Alpha-based: True, Metric: magnitude, Blockwise: False, alpha_reverse: False

In [ ]:
log_info = {'model.layers.0.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.0.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.0.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.0.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.0.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.0.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.0.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.1.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.1.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.1.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.1.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.1.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.1.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.1.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.2.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.2.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.2.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.2.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.2.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.2.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.2.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.3.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.3.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.3.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.3.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.3.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.3.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.3.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.4.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.4.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.4.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.4.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.4.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.4.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.4.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.5.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.5.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.5.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.5.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.5.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.5.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.5.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.6.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.6.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.6.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.6.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.6.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.6.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.6.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.7.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.7.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.7.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.7.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.7.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.7.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.7.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.8.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.8.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.8.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.8.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.8.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.8.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.8.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.9.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.9.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.9.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.9.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.9.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.9.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.9.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.10.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.10.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.10.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.10.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.10.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.10.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.10.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.11.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.11.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.11.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.11.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.11.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.11.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.11.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.12.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.12.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.12.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.12.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.12.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.12.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.12.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.13.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.13.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.13.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.13.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.13.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.13.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.13.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.14.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.14.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.14.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.14.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.14.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.14.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.14.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.15.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.15.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.15.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.15.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.15.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.15.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.15.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.16.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.16.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.16.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.16.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.16.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.16.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.16.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.17.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.17.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.17.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.17.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.17.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.17.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.17.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.18.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.18.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.18.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.18.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.18.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.18.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.18.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.19.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.19.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.19.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.19.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.19.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.19.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.19.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.20.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.20.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.20.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.20.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.20.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.20.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.20.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.21.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.21.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.21.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.21.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.21.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.21.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.21.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.22.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.22.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.22.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.22.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.22.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.22.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.22.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.23.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.23.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.23.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.23.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.23.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.23.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.23.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.24.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.24.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.24.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.24.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.24.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.24.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.24.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.25.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.25.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.25.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.25.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.25.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.25.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.25.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.26.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.26.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.26.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.26.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.26.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.26.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.26.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}, 'model.layers.27.self_attn.q_proj': {'applied_sparsity': np.float64(0.1468138995644844), 'metric': 'magnitude'}, 'model.layers.27.self_attn.k_proj': {'applied_sparsity': np.float64(0.1517628830242994), 'metric': 'magnitude'}, 'model.layers.27.self_attn.v_proj': {'applied_sparsity': np.float64(0.20426066484623853), 'metric': 'magnitude'}, 'model.layers.27.self_attn.o_proj': {'applied_sparsity': np.float64(0.14707123204663142), 'metric': 'magnitude'}, 'model.layers.27.mlp.gate_proj': {'applied_sparsity': np.float64(0.25220474627720335), 'metric': 'magnitude'}, 'model.layers.27.mlp.up_proj': {'applied_sparsity': np.float64(0.25613539313094347), 'metric': 'magnitude'}, 'model.layers.27.mlp.down_proj': {'applied_sparsity': np.float64(0.24175118111019953), 'metric': 'magnitude'}}


In [ ]:
import re
import numpy as np
import pandas as pd


def make_alpha_sparsity_df(log_info, results_df):
    """
    alpha_prune_llama の log_info と results_df を対応させて、
    alpha vs applied_sparsity の DataFrame を作る。
    """

    # results_df を name index にする
    if "name" in results_df.columns:
        results_indexed = results_df.set_index("name")
    else:
        results_indexed = results_df.copy()

    rows = []

    for layer_name, info in log_info.items():

        applied_sparsity = float(info["applied_sparsity"])

        alpha = np.nan
        matched_name = None

        # 1. 完全一致
        if layer_name in results_indexed.index:
            alpha = results_indexed.loc[layer_name, "alpha"]
            matched_name = layer_name

        else:
            # 2. suffix一致
            candidates = [
                name for name in results_indexed.index
                if layer_name.endswith(name) or name.endswith(layer_name)
            ]

            # 3. それでも見つからない場合、末尾の module 名で弱く探索
            # 例: self_attn.q_proj, mlp.up_proj など
            if len(candidates) == 0:
                suffix = ".".join(layer_name.split(".")[-2:])
                candidates = [
                    name for name in results_indexed.index
                    if name.endswith(suffix)
                ]

            if len(candidates) > 0:
                matched_name = candidates[0]
                alpha = results_indexed.loc[matched_name, "alpha"]

        # block index を取り出す
        m = re.search(r"layers\.(\d+)", layer_name)
        block_idx = int(m.group(1)) if m is not None else np.nan

        # layer type を取り出す
        # q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj など
        layer_type = layer_name.split(".")[-1]

        rows.append({
            "layer_name": layer_name,
            "matched_name": matched_name,
            "block_idx": block_idx,
            "layer_type": layer_type,
            "alpha": float(alpha) if not pd.isna(alpha) else np.nan,
            "applied_sparsity": applied_sparsity,
        })

    plot_df = pd.DataFrame(rows)

    return plot_df

In [ ]:
import matplotlib.pyplot as plt

plot_df = make_alpha_sparsity_df(
    log_info=log_info,
    results_df=results,
)


plt.figure(figsize=(8, 6))

for layer_type, df_sub in plot_df.groupby("layer_type"):
    plt.scatter(
        df_sub["alpha"],
        df_sub["applied_sparsity"],
        alpha=0.7,
        label=layer_type,
    )

plt.xlabel("alpha")
plt.ylabel("applied sparsity")
plt.title("Alpha vs Applied Sparsity by Layer Type Target Sparsity: 20.0%, Alpha-based: True, Metric: magnitude, Blockwise: False, alpha_reverse: False")
plt.legend()
plt.grid(True)
plt.show()

### Target Sparsity: 20.0%, Alpha-based: True, Metric: magnitude, Blockwise: False, alpha_reverse: True

In [ ]:
log_info = {'model.layers.0.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.0.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.0.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.0.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.0.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.0.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.0.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.1.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.1.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.1.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.1.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.1.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.1.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.1.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.2.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.2.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.2.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.2.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.2.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.2.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.2.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.3.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.3.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.3.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.3.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.3.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.3.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.3.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.4.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.4.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.4.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.4.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.4.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.4.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.4.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.5.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.5.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.5.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.5.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.5.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.5.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.5.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.6.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.6.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.6.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.6.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.6.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.6.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.6.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.7.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.7.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.7.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.7.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.7.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.7.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.7.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.8.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.8.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.8.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.8.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.8.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.8.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.8.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.9.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.9.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.9.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.9.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.9.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.9.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.9.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.10.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.10.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.10.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.10.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.10.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.10.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.10.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.11.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.11.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.11.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.11.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.11.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.11.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.11.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.12.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.12.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.12.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.12.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.12.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.12.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.12.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.13.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.13.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.13.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.13.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.13.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.13.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.13.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.14.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.14.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.14.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.14.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.14.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.14.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.14.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.15.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.15.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.15.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.15.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.15.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.15.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.15.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.16.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.16.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.16.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.16.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.16.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.16.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.16.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.17.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.17.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.17.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.17.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.17.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.17.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.17.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.18.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.18.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.18.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.18.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.18.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.18.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.18.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.19.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.19.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.19.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.19.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.19.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.19.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.19.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.20.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.20.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.20.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.20.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.20.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.20.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.20.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.21.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.21.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.21.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.21.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.21.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.21.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.21.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.22.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.22.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.22.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.22.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.22.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.22.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.22.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.23.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.23.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.23.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.23.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.23.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.23.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.23.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.24.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.24.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.24.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.24.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.24.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.24.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.24.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.25.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.25.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.25.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.25.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.25.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.25.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.25.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.26.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.26.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.26.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.26.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.26.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.26.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.26.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}, 'model.layers.27.self_attn.q_proj': {'applied_sparsity': np.float64(0.2570221281985802), 'metric': 'magnitude'}, 'model.layers.27.self_attn.k_proj': {'applied_sparsity': np.float64(0.2486406436358732), 'metric': 'magnitude'}, 'model.layers.27.self_attn.v_proj': {'applied_sparsity': np.float64(0.18473660087032284), 'metric': 'magnitude'}, 'model.layers.27.self_attn.o_proj': {'applied_sparsity': np.float64(0.25657241317752827), 'metric': 'magnitude'}, 'model.layers.27.mlp.gate_proj': {'applied_sparsity': np.float64(0.14961820295695993), 'metric': 'magnitude'}, 'model.layers.27.mlp.up_proj': {'applied_sparsity': np.float64(0.14732216603864912), 'metric': 'magnitude'}, 'model.layers.27.mlp.down_proj': {'applied_sparsity': np.float64(0.1560878451220865), 'metric': 'magnitude'}}


In [ ]:
import matplotlib.pyplot as plt

plot_df = make_alpha_sparsity_df(
    log_info=log_info,
    results_df=results,
)


plt.figure(figsize=(8, 6))

for layer_type, df_sub in plot_df.groupby("layer_type"):
    plt.scatter(
        df_sub["alpha"],
        df_sub["applied_sparsity"],
        alpha=0.7,
        label=layer_type,
    )

plt.xlabel("alpha")
plt.ylabel("applied sparsity")
plt.title("Alpha vs Applied Sparsity by Layer Type Target Sparsity: 20.0%, Alpha-based: True, Metric: magnitude, Blockwise: False, alpha_reverse: True")
plt.legend()
plt.grid(True)
plt.show()

# LoRA

In [ ]:
!pip uninstall -y torchao
!pip install -q -U peft accelerate

In [ ]:
import importlib.util
import torch
import transformers
import peft
import accelerate

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("torchao installed:", importlib.util.find_spec("torchao") is not None)

In [ ]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

from funcs1 import get_ppl

In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = "meta-llama/Llama-3.2-3B"

# PPL評価条件
PPL_DATASET_NAME = "wikitext2"
PPL_SEQ_LEN = 1024
PPL_BATCH_SIZE = 2

# LoRA学習条件
TRAIN_SEQ_LEN = 1024
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4

PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

OUTPUT_ROOT = "/content/lora_recovery_results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Llama tokenizerにはpad_tokenがない場合があるため設定
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)

In [ ]:
def prepare_wikitext2_train_dataset(
    tokenizer,
    seq_len=1024,
    num_proc=2,
):
    print("WikiText-2 train splitを読み込んでいます...")

    raw_train = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="train",
    )

    # 空行を除去
    raw_train = raw_train.filter(
        lambda example: example["text"] is not None
        and len(example["text"].strip()) > 0
    )

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            add_special_tokens=False,
        )

    tokenized = raw_train.map(
        tokenize_function,
        batched=True,
        remove_columns=raw_train.column_names,
        num_proc=num_proc,
        desc="Tokenizing WikiText-2 train",
    )

    def group_texts(examples):
        # batched map内のtoken列を連結
        concatenated = {
            key: sum(examples[key], [])
            for key in examples.keys()
        }

        total_length = len(concatenated["input_ids"])

        # 端数を切り捨てる
        total_length = (total_length // seq_len) * seq_len

        result = {
            key: [
                values[i:i + seq_len]
                for i in range(0, total_length, seq_len)
            ]
            for key, values in concatenated.items()
        }

        return result

    lm_train_dataset = tokenized.map(
        group_texts,
        batched=True,
        num_proc=num_proc,
        desc=f"Grouping into {seq_len}-token blocks",
    )

    print("学習系列数:", len(lm_train_dataset))

    return lm_train_dataset

In [ ]:
train_dataset = prepare_wikitext2_train_dataset(
    tokenizer=tokenizer,
    seq_len=TRAIN_SEQ_LEN,
    num_proc=2,
)

print(train_dataset)
print("最初の系列長:", len(train_dataset[0]["input_ids"]))

In [ ]:
def load_fresh_model():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="cpu",
    )

    model.config.use_cache = False

    return model

In [ ]:
def build_lra_model(
    strategy,
    max_lra_layers,
):
    """
    strategy:
      - "alpha_ascending"
      - "KS_postDE_1_descending"
    """

    model = load_fresh_model()

    if strategy == "alpha_ascending":
        lra_list = (
            results
            .sort_values(by="alpha", ascending=True)["name"]
            .tolist()
        )

    elif strategy == "KS_postDE_1_descending":
        lra_list = (
            results
            .sort_values(by="KS_postDE_1", ascending=False)["name"]
            .tolist()
        )

    else:
        raise ValueError(f"Unknown LRA strategy: {strategy}")

    print("=" * 80)
    print("LRA strategy:", strategy)
    print("LRA layers:", max_lra_layers)
    print("=" * 80)

    # run_lra_experimentはmodelをin-placeで更新する前提
    history = funcs1.run_lra_experiment(
        model=model,
        tokenizer=tokenizer,
        results_df=results,
        lra_list=lra_list,
        max_lra_layers=max_lra_layers,
        dataset_name=PPL_DATASET_NAME,
        DE=True,
        seq_len=PPL_SEQ_LEN,
        batch_size=PPL_BATCH_SIZE,
    )

    selected_layers = lra_list[:max_lra_layers]

    print("LRA完了")
    print("圧縮層数:", len(selected_layers))

    return model, history, selected_layers

In [ ]:
def build_pruned_model(
    strategy,
    target_sparsity,
):
    """
    strategy:
      - "uniform_magnitude"
      - "alpha_reverse_magnitude"
    """

    model = load_fresh_model()

    if strategy == "uniform_magnitude":
        alpha_prune = False
        alpha_reverse = False

    elif strategy == "alpha_reverse_magnitude":
        alpha_prune = True
        alpha_reverse = True

    else:
        raise ValueError(f"Unknown pruning strategy: {strategy}")

    print("=" * 80)
    print("Pruning strategy:", strategy)
    print("Target sparsity:", target_sparsity)
    print("=" * 80)

    # run_pruning_experimentはmodelの重みをin-placeで枝刈りする前提
    pruning_result = funcs1.run_pruning_experiment(
        model=model,
        tokenizer=tokenizer,
        results_df=results,
        dataset_name=PPL_DATASET_NAME,
        seq_len=PPL_SEQ_LEN,
        batch_size=PPL_BATCH_SIZE,
        sparsity=target_sparsity,
        alpha_prune=alpha_prune,
        prune_metric="magnitude",
        blockwise=False,
        alpha_reverse=alpha_reverse,
    )

    print("Pruning完了")
    print(pruning_result)

    return model, pruning_result

In [ ]:
def check_model_finite(model):
    bad_parameters = []

    for name, param in model.named_parameters():
        if not torch.is_floating_point(param):
            continue

        if not torch.isfinite(param).all():
            bad_parameters.append({
                "name": name,
                "nan": torch.isnan(param).sum().item(),
                "inf": torch.isinf(param).sum().item(),
                "shape": tuple(param.shape),
                "dtype": str(param.dtype),
            })

    if bad_parameters:
        print("❌ NaN/Infを含むパラメータがあります")
        for item in bad_parameters[:20]:
            print(item)
        raise RuntimeError("Model contains NaN/Inf")

    print("✅ 全パラメータはfiniteです")


In [ ]:
TARGET_MODULE_SUFFIXES = (
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
)


def count_base_weight_sparsity(model):
    total = 0
    zeros = 0

    for name, module in model.named_modules():
        if not name.endswith(TARGET_MODULE_SUFFIXES):
            continue

        if not hasattr(module, "weight"):
            continue

        weight = module.weight.detach()

        total += weight.numel()
        zeros += (weight == 0).sum().item()

    sparsity = zeros / total if total > 0 else 0.0

    return {
        "zero_weights": zeros,
        "total_weights": total,
        "sparsity": sparsity,
    }

In [ ]:
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


def attach_lora(model):
    model.config.use_cache = False

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=LORA_TARGET_MODULES,
    )

    peft_model = get_peft_model(
        model,
        lora_config,
    )

    # メモリ削減
    peft_model.gradient_checkpointing_enable()
    peft_model.enable_input_require_grads()

    peft_model.print_trainable_parameters()

    return peft_model

In [ ]:
def evaluate_wikitext2_ppl(model):
    model.eval()

    ppl = get_ppl(
        model,
        tokenizer,
        dataset_name=PPL_DATASET_NAME,
        seq_len=PPL_SEQ_LEN,
        batch_size=PPL_BATCH_SIZE,
    )

    return float(ppl)

In [ ]:
def train_lora_adapter(
    model,
    experiment_name,
    train_dataset,
):
    output_dir = os.path.join(
        OUTPUT_ROOT,
        experiment_name,
    )

    os.makedirs(output_dir, exist_ok=True)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    training_args = TrainingArguments(
        output_dir=output_dir,

        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,

        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

        fp16=True,
        bf16=False,

        gradient_checkpointing=True,

        logging_strategy="steps",
        logging_steps=10,

        save_strategy="epoch",
        save_total_limit=1,

        report_to="none",

        optim="adamw_torch",
        weight_decay=0.0,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",

        max_grad_norm=1.0,

        remove_unused_columns=False,
        dataloader_num_workers=2,

        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
    )

    print("=" * 80)
    print("LoRA fine-tuning開始:", experiment_name)
    print("=" * 80)

    train_result = trainer.train()

    # Adapterのみ保存
    adapter_dir = os.path.join(
        output_dir,
        "final_adapter",
    )

    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    train_metrics = dict(train_result.metrics)

    with open(
        os.path.join(output_dir, "train_metrics.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            train_metrics,
            f,
            ensure_ascii=False,
            indent=2,
        )

    return model, trainer, train_metrics

In [ ]:
def run_lora_recovery_experiment(
    experiment_name,
    compression_type,
    compression_strategy,
    train_dataset,
    max_lra_layers=None,
    target_sparsity=None,
):
    """
    compression_type:
      - "lra"
      - "pruning"
    """

    print("\n" + "#" * 100)
    print("Experiment:", experiment_name)
    print("#" * 100)

    # ---------------------------------------------------------
    # 1. 圧縮モデルをfresh modelから再構築
    # ---------------------------------------------------------
    compression_info = {}

    if compression_type == "lra":
        if max_lra_layers is None:
            raise ValueError("LRAではmax_lra_layersが必要です")

        model, lra_history, selected_layers = build_lra_model(
            strategy=compression_strategy,
            max_lra_layers=max_lra_layers,
        )

        compression_info = {
            "compression_type": "lra",
            "compression_strategy": compression_strategy,
            "num_lra_layers": max_lra_layers,
            "selected_layers": selected_layers,
        }

    elif compression_type == "pruning":
        if target_sparsity is None:
            raise ValueError("Pruningではtarget_sparsityが必要です")

        model, pruning_result = build_pruned_model(
            strategy=compression_strategy,
            target_sparsity=target_sparsity,
        )

        compression_info = {
            "compression_type": "pruning",
            "compression_strategy": compression_strategy,
            "target_sparsity": target_sparsity,
            "pruning_result": pruning_result,
        }

    else:
        raise ValueError(
            f"Unknown compression_type: {compression_type}"
        )

    # ---------------------------------------------------------
    # 2. 数値チェック
    # ---------------------------------------------------------
    check_model_finite(model)

    sparsity_before_lora = None

    if compression_type == "pruning":
        sparsity_before_lora = count_base_weight_sparsity(model)
        print("LoRA前のbase-weight sparsity:", sparsity_before_lora)

    # ---------------------------------------------------------
    # 3. LoRA前のPPL
    # ---------------------------------------------------------
    print("LoRA前PPLを測定します")

    ppl_before_lora = evaluate_wikitext2_ppl(model)

    print(
        f"✅ {experiment_name} "
        f"LoRA前 PPL = {ppl_before_lora:.4f}"
    )

    # ---------------------------------------------------------
    # 4. LoRA adapterを追加
    # ---------------------------------------------------------
    model = attach_lora(model)

    # ---------------------------------------------------------
    # 5. LoRA学習
    # ---------------------------------------------------------
    model, trainer, train_metrics = train_lora_adapter(
        model=model,
        experiment_name=experiment_name,
        train_dataset=train_dataset,
    )

    # ---------------------------------------------------------
    # 6. LoRA後のPPL
    # ---------------------------------------------------------
    print("LoRA後PPLを測定します")

    ppl_after_lora = evaluate_wikitext2_ppl(model)

    print(
        f"✅ {experiment_name} "
        f"LoRA後 PPL = {ppl_after_lora:.4f}"
    )

    # ---------------------------------------------------------
    # 7. Pruningモデルのゼロが維持されているか確認
    # ---------------------------------------------------------
    sparsity_after_lora = None

    if compression_type == "pruning":
        sparsity_after_lora = count_base_weight_sparsity(model)

        print(
            "LoRA後のbase-weight sparsity:",
            sparsity_after_lora,
        )

    # ---------------------------------------------------------
    # 8. 回復率
    # ---------------------------------------------------------
    baseline_ppl = 8.740511832292928

    degradation_before = ppl_before_lora - baseline_ppl
    degradation_after = ppl_after_lora - baseline_ppl

    if degradation_before > 0:
        recovery_ratio = (
            degradation_before - degradation_after
        ) / degradation_before
    else:
        recovery_ratio = np.nan

    summary = {
        "experiment_name": experiment_name,
        "compression_type": compression_type,
        "compression_strategy": compression_strategy,

        "ppl_baseline": baseline_ppl,
        "ppl_before_lora": ppl_before_lora,
        "ppl_after_lora": ppl_after_lora,

        "absolute_ppl_improvement":
            ppl_before_lora - ppl_after_lora,

        "recovery_ratio":
            recovery_ratio,

        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "train_seq_len": TRAIN_SEQ_LEN,

        "sparsity_before_lora":
            None if sparsity_before_lora is None
            else sparsity_before_lora["sparsity"],

        "sparsity_after_lora":
            None if sparsity_after_lora is None
            else sparsity_after_lora["sparsity"],
    }

    summary.update({
        key: value
        for key, value in compression_info.items()
        if key not in ["selected_layers", "pruning_result"]
    })

    experiment_dir = os.path.join(
        OUTPUT_ROOT,
        experiment_name,
    )

    with open(
        os.path.join(experiment_dir, "summary.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            summary,
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    print("\nSummary:")
    for key, value in summary.items():
        print(f"{key}: {value}")

    # ---------------------------------------------------------
    # 9. メモリ解放前にtrainer参照を削除
    # ---------------------------------------------------------
    del trainer

    return summary, model

In [ ]:
EXPERIMENTS = [
    {
        "experiment_name":
            "lra_alpha_ascending_20layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "alpha_ascending",

        "max_lra_layers":
            20,
    },

    {
        "experiment_name":
            "pruning_uniform_magnitude_target005",

        "compression_type":
            "pruning",

        "compression_strategy":
            "uniform_magnitude",

        "target_sparsity":
            0.05,
    },

    {
        "experiment_name":
            "lra_KS_postDE_1_descending_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "KS_postDE_1_descending",

        "max_lra_layers":
            50,
    },

    {
        "experiment_name":
            "pruning_alpha_reverse_magnitude_target010",

        "compression_type":
            "pruning",

        "compression_strategy":
            "alpha_reverse_magnitude",

        "target_sparsity":
            0.10,
    },
]

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

test_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu",
)

test_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

test_model = get_peft_model(test_model, test_config)
test_model.print_trainable_parameters()

In [ ]:
all_summaries = []

for exp in EXPERIMENTS:
    try:
        summary, trained_model = run_lora_recovery_experiment(
            experiment_name=exp["experiment_name"],
            compression_type=exp["compression_type"],
            compression_strategy=exp["compression_strategy"],
            train_dataset=train_dataset,
            max_lra_layers=exp.get("max_lra_layers"),
            target_sparsity=exp.get("target_sparsity"),
        )

        all_summaries.append(summary)

    except Exception as e:
        print(
            f"❌ Experiment failed: "
            f"{exp['experiment_name']}"
        )
        print(type(e).__name__, e)

        all_summaries.append({
            "experiment_name":
                exp["experiment_name"],
            "status":
                "failed",
            "error":
                repr(e),
        })

    finally:
        if "trained_model" in locals():
            del trained_model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        # 途中までの結果も毎回保存
        pd.DataFrame(all_summaries).to_csv(
            os.path.join(
                OUTPUT_ROOT,
                "lora_recovery_summary.csv",
            ),
            index=False,
        )

        print("GPUメモリを解放しました")


In [ ]:
summary_df = pd.DataFrame(all_summaries)

display_columns = [
    "experiment_name",
    "compression_type",
    "compression_strategy",
    "ppl_baseline",
    "ppl_before_lora",
    "ppl_after_lora",
    "absolute_ppl_improvement",
    "recovery_ratio",
    "sparsity_before_lora",
    "sparsity_after_lora",
]

existing_columns = [
    col
    for col in display_columns
    if col in summary_df.columns
]

display(summary_df[existing_columns])

In [ ]:
summary_path = os.path.join(
    OUTPUT_ROOT,
    "lora_recovery_summary.csv",
)

summary_df.to_csv(
    summary_path,
    index=False,
)

print("保存先:", summary_path)

In [ ]:
ADDITIONAL_EXPERIMENTS = [
    {
        "experiment_name":
            "lra_KS_postDE_1_descending_100layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "KS_postDE_1_descending",

        "max_lra_layers":
            100,
    },

    {
        "experiment_name":
            "pruning_uniform_magnitude_target030",

        "compression_type":
            "pruning",

        "compression_strategy":
            "uniform_magnitude",

        "target_sparsity":
            0.30,
    },
]

In [ ]:
additional_summaries = []

for exp in ADDITIONAL_EXPERIMENTS:
    trained_model = None

    try:
        summary, trained_model = run_lora_recovery_experiment(
            experiment_name=exp["experiment_name"],
            compression_type=exp["compression_type"],
            compression_strategy=exp["compression_strategy"],
            train_dataset=train_dataset,
            max_lra_layers=exp.get("max_lra_layers"),
            target_sparsity=exp.get("target_sparsity"),
        )

        summary["status"] = "completed"
        additional_summaries.append(summary)

    except Exception as e:
        print(
            f"❌ Experiment failed: "
            f"{exp['experiment_name']}"
        )
        print(type(e).__name__, e)

        additional_summaries.append({
            "experiment_name": exp["experiment_name"],
            "compression_type": exp["compression_type"],
            "compression_strategy": exp["compression_strategy"],
            "status": "failed",
            "error": repr(e),
        })

    finally:
        if trained_model is not None:
            del trained_model
            trained_model = None

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        # 途中経過を毎回保存
        additional_df = pd.DataFrame(additional_summaries)

        additional_df.to_csv(
            os.path.join(
                OUTPUT_ROOT,
                "lora_recovery_additional_summary.csv",
            ),
            index=False,
        )

        print("GPUメモリを解放しました")

In [ ]:
def run_uncompressed_lora_control(
    experiment_name="uncompressed_lora_control",
):
    model = load_fresh_model()

    check_model_finite(model)

    print("LoRA前PPLを測定します")
    ppl_before_lora = evaluate_wikitext2_ppl(model)

    print(
        f"✅ 未圧縮モデル LoRA前 PPL = "
        f"{ppl_before_lora:.4f}"
    )

    model = attach_lora(model)

    model, trainer, train_metrics = train_lora_adapter(
        model=model,
        experiment_name=experiment_name,
        train_dataset=train_dataset,
    )

    print("LoRA後PPLを測定します")
    ppl_after_lora = evaluate_wikitext2_ppl(model)

    print(
        f"✅ 未圧縮モデル LoRA後 PPL = "
        f"{ppl_after_lora:.4f}"
    )

    summary = {
        "experiment_name": experiment_name,
        "compression_type": "none",
        "compression_strategy": "uncompressed",
        "ppl_baseline": 8.740511832292928,
        "ppl_before_lora": ppl_before_lora,
        "ppl_after_lora": ppl_after_lora,
        "absolute_ppl_improvement":
            ppl_before_lora - ppl_after_lora,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "train_seq_len": TRAIN_SEQ_LEN,
    }

    print("\nSummary:")
    for key, value in summary.items():
        print(f"{key}: {value}")

    del trainer

    return summary, model


In [ ]:
control_summary, control_model = (
    run_uncompressed_lora_control()
)

## DE vs OffDE

In [ ]:
def train_lora_adapter(
    model,
    experiment_name,
    train_dataset,
):
    output_dir = os.path.join(
        OUTPUT_ROOT,
        experiment_name,
    )

    os.makedirs(output_dir, exist_ok=True)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    training_args = TrainingArguments(
        output_dir=output_dir,

        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,

        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

        fp16=True,
        bf16=False,

        gradient_checkpointing=True,

        logging_strategy="steps",
        logging_steps=10,

        save_strategy="epoch",
        save_total_limit=1,

        report_to="none",

        optim="adamw_torch",
        weight_decay=0.0,
        # warmup_ratio=0.03,
        lr_scheduler_type="cosine",

        max_grad_norm=1.0,

        remove_unused_columns=False,
        dataloader_num_workers=2,

        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
    )

    print("=" * 80)
    print("LoRA fine-tuning開始:", experiment_name)
    print("=" * 80)

    train_result = trainer.train()

    # Adapterのみ保存
    adapter_dir = os.path.join(
        output_dir,
        "final_adapter",
    )

    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    train_metrics = dict(train_result.metrics)

    with open(
        os.path.join(output_dir, "train_metrics.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            train_metrics,
            f,
            ensure_ascii=False,
            indent=2,
        )

    return model, trainer, train_metrics

In [ ]:
def build_lra_model(
    strategy,
    max_lra_layers,
):
    """
    strategy:
      - "alpha_ascending_DE"
      - "alpha_ascending_OffDE"
      - "KS_postDE_1_descending_DE"
      - "KS_postDE_1_descending_OffDE"
    """

    model = load_fresh_model()

    if strategy == "alpha_ascending_DE":
        lra_list = (
            results
            .sort_values(by="alpha", ascending=True)["name"]
            .tolist()
        )
        DE = True

    elif strategy == "KS_postDE_1_descending_DE":
        lra_list = (
            results
            .sort_values(by="KS_postDE_1", ascending=False)["name"]
            .tolist()
        )
        DE = True

    elif strategy == "alpha_ascending_OffDE":
        lra_list = (
            results
            .sort_values(by="alpha", ascending=True)["name"]
            .tolist()
        )
        DE = False

    elif strategy == "KS_postDE_1_descending_OffDE":
        lra_list = (
            results
            .sort_values(by="KS_postDE_1", ascending=False)["name"]
            .tolist()
        )
        DE = False

    else:
        raise ValueError(f"Unknown LRA strategy: {strategy}")

    print("=" * 80)
    print("LRA strategy:", strategy)
    print("LRA layers:", max_lra_layers)
    print("=" * 80)

    # run_lra_experimentはmodelをin-placeで更新する前提
    history = funcs1.run_lra_experiment(
        model=model,
        tokenizer=tokenizer,
        results_df=results,
        lra_list=lra_list,
        max_lra_layers=max_lra_layers,
        dataset_name=PPL_DATASET_NAME,
        DE=DE,
        PPLcalc = False,
        seq_len=PPL_SEQ_LEN,
        batch_size=PPL_BATCH_SIZE,
    )

    selected_layers = lra_list[:max_lra_layers]

    print("LRA完了")
    print("圧縮層数:", len(selected_layers))

    return model, history, selected_layers

In [ ]:
EXPERIMENTS = [
    {
        "experiment_name":
            "lra_alpha_ascending_DE_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "alpha_ascending_DE",

        "max_lra_layers":
            50,
    },
    {
        "experiment_name":
            "lra_alpha_ascending_OffDE_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "alpha_ascending_OffDE",

        "max_lra_layers":
            50,
    },

    {
        "experiment_name":
            "lra_KS_postDE_1_descending_DE_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "KS_postDE_1_descending_DE",

        "max_lra_layers":
            50,
    },
    {
        "experiment_name":
            "lra_KS_postDE_1_descending_OffDE_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "KS_postDE_1_descending_OffDE",

        "max_lra_layers":
            50,
    },


]

In [ ]:
all_summaries = []

for exp in EXPERIMENTS:
    try:
        summary, trained_model = run_lora_recovery_experiment(
            experiment_name=exp["experiment_name"],
            compression_type=exp["compression_type"],
            compression_strategy=exp["compression_strategy"],
            train_dataset=train_dataset,
            max_lra_layers=exp.get("max_lra_layers"),
            target_sparsity=exp.get("target_sparsity"),
        )

        all_summaries.append(summary)

    except Exception as e:
        print(
            f"❌ Experiment failed: "
            f"{exp['experiment_name']}"
        )
        print(type(e).__name__, e)

        all_summaries.append({
            "experiment_name":
                exp["experiment_name"],
            "status":
                "failed",
            "error":
                repr(e),
        })

    finally:
        if "trained_model" in locals():
            del trained_model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        # 途中までの結果も毎回保存
        pd.DataFrame(all_summaries).to_csv(
            os.path.join(
                OUTPUT_ROOT,
                "lora_recovery_summary.csv",
            ),
            index=False,
        )

        print("GPUメモリを解放しました")


In [ ]:
import pandas as pd
summary_df = pd.DataFrame(all_summaries)

display_columns = [
    "experiment_name",
    "compression_type",
    "compression_strategy",
    "ppl_baseline",
    "ppl_before_lora",
    "ppl_after_lora",
    "absolute_ppl_improvement",
    "recovery_ratio",
    "sparsity_before_lora",
    "sparsity_after_lora",
]

existing_columns = [
    col
    for col in display_columns
    if col in summary_df.columns
]

display(summary_df[existing_columns])


In [ ]:
from google.colab import runtime

runtime.unassign()

In [ ]:
EXPERIMENTS = [

    {
        "experiment_name":
            "lra_KS_postDE_1_descending_OffDE_50layers",

        "compression_type":
            "lra",

        "compression_strategy":
            "KS_postDE_1_descending_OffDE",

        "max_lra_layers":
            50,
    },


]

In [ ]:
all_summaries = []

for exp in EXPERIMENTS:
    try:
        summary, trained_model = run_lora_recovery_experiment(
            experiment_name=exp["experiment_name"],
            compression_type=exp["compression_type"],
            compression_strategy=exp["compression_strategy"],
            train_dataset=train_dataset,
            max_lra_layers=exp.get("max_lra_layers"),
            target_sparsity=exp.get("target_sparsity"),
        )

        all_summaries.append(summary)

    except Exception as e:
        print(
            f"❌ Experiment failed: "
            f"{exp['experiment_name']}"
        )
        print(type(e).__name__, e)

        all_summaries.append({
            "experiment_name":
                exp["experiment_name"],
            "status":
                "failed",
            "error":
                repr(e),
        })

    finally:
        if "trained_model" in locals():
            del trained_model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        # 途中までの結果も毎回保存
        pd.DataFrame(all_summaries).to_csv(
            os.path.join(
                OUTPUT_ROOT,
                "lora_recovery_summary.csv",
            ),
            index=False,
        )

        print("GPUメモリを解放しました")


# データ

In [ ]:
import wandb
import pandas as pd

# ご自身のユーザー名/プロジェクト名に変更してください
PROJECT_PATH = "ryoya-zushi1210-tokyo-university-of-science/LRA-Ablation-Study-Updated"

api = wandb.Api()
runs = api.runs(PROJECT_PATH)

all_data = []
for run in runs:
    # Runの名前（例: alpha_ascending など）を取得
    run_name = run.name

    # このRunの全履歴（step, ppl, reduction_ratio など）を取得
    history = run.history()
    history['run_name'] = run_name # どのRunのデータか分かるように列を追加

    all_data.append(history)

# すべてのRunのデータを結合してCSVに保存
final_df = pd.concat(all_data, ignore_index=True)
final_df.to_csv("wandb_export_data.csv", index=False)
print("✅ wandb_export_data.csv を保存しました！")

In [ ]:
import wandb
import pandas as pd
import os

# ==========================================
# 1. ここをご自身の情報に書き換えてください
# ==========================================
# 例: "ryoya/LRA-Ablation-Study-Updated"
PROJECT_PATH = "ryoya-zushi1210-tokyo-university-of-science/LRA-Ablation-Study-Updated"

# ==========================================
# 2. WandB から全履歴データをダウンロード
# ==========================================
print(f"プロジェクト '{PROJECT_PATH}' からデータを取得中...")
api = wandb.Api()
runs = api.runs(PROJECT_PATH)

all_history_data = []

for run in runs:
    print(f"Run: {run.name} の履歴を取得中...")

    # history() を呼ぶことで、ステップごとの推移をすべて取得できます
    # (デフォルトで最大500件まで。100層なら余裕で収まります)
    history_df = run.history()

    if history_df.empty:
        print("  -> データなし、スキップします。")
        continue

    # どのRunのデータか識別できるように列を追加
    history_df['run_name'] = run.name
    history_df['method'] = run.config.get('method', 'unknown')
    history_df['DE_status'] = run.config.get('DE', 'unknown')

    all_history_data.append(history_df)

# ==========================================
# 3. CSVとして保存
# ==========================================
if all_history_data:
    final_df = pd.concat(all_history_data, ignore_index=True)

    # カラムの順番を見やすく整理（存在しないカラムは無視）
    cols_order = ['run_name', 'method', 'DE_status', 'step', 'layer_name', 'reduction_ratio_percent', 'ppl_wikitext2', 'alpha_val']
    existing_cols = [c for c in cols_order if c in final_df.columns]
    other_cols = [c for c in final_df.columns if c not in existing_cols]
    final_df = final_df[existing_cols + other_cols]

    # 保存
    csv_filename = "lra_full_history_data.csv"
    final_df.to_csv(csv_filename, index=False)

    print(f"\n✅ 成功！全ステップの履歴データを '{csv_filename}' に保存しました。")
    print(f"総データ行数: {len(final_df)}行")
else:
    print("データが取得できませんでした。プロジェクト名などを確認してください。")

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 適宜パスを変更
CSV_PATH = "lra_full_history_data.csv"

df = pd.read_csv(CSV_PATH)

print("データ形状:", df.shape)
print("列名:", df.columns.tolist())

In [ ]:
TARGET_RUNS = {
    ("alpha_ascending", True):
        "alpha_ascending_2_with_params",

    ("alpha_ascending", False):
        "alpha_ascending_OffDE_updated",

    ("s_hat_ratio_postDE_descending", True):
        "s_hat_ratio_postDE_descending_with_params",

    ("s_hat_ratio_postDE_descending", False):
        "s_hat_ratio_postDE_descending_OffDE_updated",

    ("KS_postDE_1_descending", True):
        "KS_postDE_1_descending_2_with_params",

    ("KS_postDE_1_descending", False):
        "KS_postDE_1_descending_OffDE_updated",
}

available_runs = set(df["run_name"].dropna().unique())

for (method, de_status), run_name in TARGET_RUNS.items():
    status = "✅" if run_name in available_runs else "❌"

    print(
        status,
        f"method={method}, "
        f"DE={de_status}, "
        f"run={run_name}"
    )

In [ ]:
def extract_layer_information(layer_name):
    """
    Hugging Face形式のlayer名から、
    Transformer block番号とmodule種類を抽出する。
    """

    if pd.isna(layer_name) or layer_name == "baseline":
        return pd.Series({
            "block_idx": np.nan,
            "module_type": "baseline",
            "module_group": "baseline",
        })

    # model.layers.14... の14を抽出
    block_match = re.search(r"layers\.(\d+)", str(layer_name))

    block_idx = (
        int(block_match.group(1))
        if block_match is not None
        else np.nan
    )

    module_type = str(layer_name).split(".")[-1]

    if module_type in {
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    }:
        module_group = "attention"

    elif module_type in {
        "gate_proj",
        "up_proj",
        "down_proj",
    }:
        module_group = "MLP"

    else:
        module_group = "other"

    return pd.Series({
        "block_idx": block_idx,
        "module_type": module_type,
        "module_group": module_group,
    })

def calculate_ppl_jumps(run_df):
    """
    1つのrunについて、各stepのPPL増加指標を計算する。

    追加される主な列
    ----------------
    ppl_previous:
        直前stepのPPL

    ppl_absolute_increase:
        PPL_t - PPL_{t-1}

    ppl_ratio:
        PPL_t / PPL_{t-1}

    ppl_percent_increase:
        (PPL_t / PPL_{t-1} - 1) * 100

    log_ppl_increase:
        log(PPL_t) - log(PPL_{t-1})
        = log(PPL_t / PPL_{t-1})

    reduction_increment:
        今回の層をLRAしたことによる削減率の増分
    """

    run_df = (
        run_df
        .sort_values("step")
        .drop_duplicates(subset="step", keep="last")
        .reset_index(drop=True)
        .copy()
    )

    # 直前step
    run_df["ppl_previous"] = (
        run_df["ppl_wikitext2"].shift(1)
    )

    run_df["reduction_previous"] = (
        run_df["reduction_ratio_percent"].shift(1)
    )

    # 絶対的増加量
    run_df["ppl_absolute_increase"] = (
        run_df["ppl_wikitext2"]
        - run_df["ppl_previous"]
    )

    # 増加倍率
    run_df["ppl_ratio"] = (
        run_df["ppl_wikitext2"]
        / run_df["ppl_previous"]
    )

    # 増加率（%）
    run_df["ppl_percent_increase"] = (
        run_df["ppl_ratio"] - 1.0
    ) * 100.0

    # 対数PPLの増分
    run_df["log_ppl"] = np.log(
        run_df["ppl_wikitext2"]
    )

    run_df["log_ppl_increase"] = (
        run_df["log_ppl"]
        - run_df["log_ppl"].shift(1)
    )

    # 累積削減率の増分
    run_df["reduction_increment"] = (
        run_df["reduction_ratio_percent"]
        - run_df["reduction_previous"]
    )

    # 層情報を付加
    layer_info = run_df["layer_name"].apply(
        extract_layer_information
    )

    run_df = pd.concat(
        [run_df, layer_info],
        axis=1,
    )

    # step=0には直前stepがないため除外
    run_df = run_df[
        run_df["step"] > 0
    ].copy()

    # 非finite値を除外
    run_df = run_df[
        np.isfinite(run_df["ppl_wikitext2"])
        & np.isfinite(run_df["ppl_previous"])
        & np.isfinite(run_df["ppl_ratio"])
    ].copy()

    return run_df


all_jump_data = []

for (method, de_status), run_name in TARGET_RUNS.items():

    run_df = df[
        df["run_name"] == run_name
    ].copy()

    if run_df.empty:
        print(f"⚠️ runが見つかりません: {run_name}")
        continue

    jump_df = calculate_ppl_jumps(run_df)

    jump_df["method_label"] = method
    jump_df["DE_label"] = (
        "DE" if de_status else "OffDE"
    )

    all_jump_data.append(jump_df)

jump_df_all = pd.concat(
    all_jump_data,
    ignore_index=True,
)

print("分析対象行数:", len(jump_df_all))

In [ ]:
TOP_K = 10

display_columns = [
    "method_label",
    "DE_label",
    "step",
    "layer_name",
    "block_idx",
    "module_group",
    "module_type",
    "ppl_previous",
    "ppl_wikitext2",
    "ppl_absolute_increase",
    "ppl_ratio",
    "ppl_percent_increase",
    "log_ppl_increase",
    "reduction_ratio_percent",
    "reduction_increment",
]

top_jumps_by_run = (
    jump_df_all
    .sort_values(
        ["run_name", "log_ppl_increase"],
        ascending=[True, False],
    )
    .groupby("run_name", group_keys=False)
    .head(TOP_K)
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", None)

display(
    top_jumps_by_run[display_columns]
)

In [ ]:
top_jumps_by_run[display_columns].to_csv("PPL_an_perrun.csv", index=False)

In [ ]:
top_overall = (
    jump_df_all
    .sort_values(
        "log_ppl_increase",
        ascending=False,
    )
    .head(100)
    .reset_index(drop=True)
)

display(
    top_overall[display_columns]
)

In [ ]:
top_overall[display_columns].to_csv("top_overall2.csv",index=False)